# Capítulo 5 — Análises dos resultados experimentais

Notebook **único** das análises do Capítulo 5. Nasce como cópia do `2. analises_dissertacao.ipynb`
(leitura do corpus, catálogo de exibição, visualização de fronts) e acrescenta as seis levas de
análise. Como o `3. clustering familias cap3.8.ipynb`, ele **gera e grava as figuras** direto em
`dissertacao/figures/capitulo5/` — o notebook é a fonte única de cômputo do capítulo.

**Proveniência.** Corpus experimental congelado em 22/08/2026 (`resultados_experimentos/`, 126 GB).
As métricas de endpoint vêm da camada oficial `src/metrics.py` (IGD+ primário por D70; HV com
objetivos normalizados pela régua S.5 congelada e ref-point 1,1 por coordenada, D69/D92), lidas do
cache `data/analysis_cache/cache_endpoint.parquet` e **reconferidas por recálculo amostral** nesta
sessão. Nenhum número entra no texto sem sair daqui.

**Estrutura**

| Parte | Leva | Seção do capítulo |
|---|---|---|
| 0 | Fundações: famílias, características dos problemas, métricas, completude | §5.1 |
| 1 | Característica do problema × característica do algoritmo | §5.2 |
| 2 | Aprofundamento: por que cada família vence onde vence | §5.3 |
| 3 | Maldição da dimensionalidade e custo | §5.4 |
| 4 | Impacto do surrogate: a sonda | §5.5 |
| 5 | Os três problemas reais | §5.6 |
| 6 | Testes estatísticos | §5.7 |

**Convenções fixas de todas as levas:** duas métricas (IGD+ ↓ e HV ↑) · duas lentes (desempenho
absoluto e diferença contra o piso casado) · comparação sempre **dentro** de cada problema (valores
absolutos não se comparam entre problemas, porque o orçamento é $31D-1$ e varia com $D$).

## 0.1 · Setup, caminhos e catálogo de exibição

Constantes transcritas do notebook 2. Acrescenta `FIGDIR` (destino das figuras do capítulo) e a
função `salvar`, que grava PDF + PNG com o mesmo nome — o `.tex` usa o PDF.

In [ ]:
import os, re, json, warnings, itertools
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.patches import Patch
warnings.filterwarnings("ignore")

MESTRADO = Path("/Users/gmello/Documents/python_repos/mestrado")
CORPUS   = MESTRADO / "resultados_experimentos"
UA       = MESTRADO / "ua-dd-saea"
DISS     = MESTRADO / "dissertacao"
CACHE    = UA / "data" / "analysis_cache"
FIGDIR   = DISS / "figures" / "capitulo5"
TABDIR   = UA / "data" / "analysis_cache" / "cap5_tabelas"
FIGDIR.mkdir(parents=True, exist_ok=True); TABDIR.mkdir(parents=True, exist_ok=True)

# ── Plantel e pisos ──────────────────────────────────────────────────────────
# 15 algoritmos assistidos (o roster do cap. 4) executados como 16 configurações
# — o b5 roda como duas (b5m = Prob-MOEA/D, b5r = Prob-RVEA) — mais os 5 pisos.
ASSISTIDOS = ['b1','b3','b4','b5m','b5r','c122','c141','c149','c154','c217',
              'c238','c262','e7','e74','e81','e103']
PISOS      = ['nsga2','nsga3','moead','smsemoa','moead_media']
ALGORITMOS = ASSISTIDOS + PISOS
OFFLINES   = {'b5m':'off','b5r':'off','e103':'off','moead_media':'off'}   # exp='off' no nome do artefato
CONFIG2ID  = {a: ('b5' if a in ('b5m','b5r') else a) for a in ASSISTIDOS}  # config -> id do corpus 127

PROBLEMAS_SINT = ["MMF1","MMF4","MMF11_L","MMF16_20","ZDT1","ZDT3","ZDT4","ZDT6",
                  "DTLZ1","DTLZ2","DTLZ3","DTLZ4","DTLZ7","WFG1","WFG2","WFG4","WFG5","WFG9",
                  "BBOB_F1","BBOB_F5","BBOB_F17","BBOB_F22","BBOB_F37","BBOB_F49","BBOB_F55"]
PROBLEMAS_REAIS = ["RE21","ESTOQUE40","DDMOP7"]
PROBLEMAS = PROBLEMAS_SINT + PROBLEMAS_REAIS
SEEDS = list(range(29)) + [42]
SEM_FRONT = {"DDMOP7"}          # D102.4: sem front de referência -> só HV

# ── Piso casado por princípio de seleção (cap. 4, §4.2.1) ────────────────────
# NSGA-II ~ dominância · MOEA/D ~ decomposição · NSGA-III ~ indicador/pontos de
# referência · SMS-EMOA ~ hipervolume. Os BO-especiais, sem análogo populacional,
# são lidos contra a BANDA (o melhor dos quatro pisos em cada problema-semente).
PISO_CASADO = {'b1':'moead','b3':'moead','b4':'nsga2','c122':'moead','c141':'nsga2',
               'c149':'BANDA','c154':'BANDA','c217':'nsga2','c238':'nsga2','c262':'smsemoa',
               'e7':'nsga3','e74':'nsga2','e81':'BANDA',
               'b5m':'moead_media','b5r':'moead_media','e103':'moead_media'}

# ── Catálogo de exibição (notebook 2): nome e cor fixos por configuração ─────
NOMES = {'b1':'ParEGO','b3':'K-RVEA','b4':'CSEA','c122':'θ-DEA-DP','c141':'MMRAEA',
         'c149':'LBN-MOBO','c154':'JES','c217':'PC-SAEA','c238':'EIM','c262':'qNEHVI',
         'e7':'EDN-ARMOEA','e74':'CLMEA','e81':'qPOTS',
         'b5m':'Prob-MOEA/D','b5r':'Prob-RVEA','e103':'IBEA-MS',
         'nsga2':'NSGA-II','nsga3':'NSGA-III','moead':'MOEA/D','smsemoa':'SMS-EMOA',
         'moead_media':'MOEA/D-média'}
CORES = {'b1':'tab:blue','b3':'tab:orange','b4':'tab:green','c122':'tab:red',
         'c141':'tab:purple','c149':'tab:brown','c154':'tab:pink','c217':'olive',
         'c238':'tab:cyan','c262':'gold','e7':'darkviolet','e74':'teal','e81':'crimson',
         'b5m':'royalblue','b5r':'mediumseagreen','e103':'darkorange',
         'nsga2':'dimgray','nsga3':'darkgray','moead':'gray','smsemoa':'silver',
         'moead_media':'lightgray'}
nome = lambda a: NOMES.get(a, a)

# ── Estilo das figuras (o mesmo do notebook 3, para o capítulo ficar coeso) ──
sns.set_theme(style="white", context="notebook")
plt.rcParams.update({"figure.dpi":110, "savefig.dpi":200, "axes.titleweight":"bold",
                     "axes.titlesize":12, "font.size":10, "axes.titlepad":10,
                     "axes.edgecolor":"#cfcfcf", "axes.linewidth":0.6,
                     "pdf.fonttype":42, "ps.fonttype":42})
GREENS = LinearSegmentedColormap.from_list("GreensTrunc", plt.get_cmap("Greens")(np.linspace(0.10,0.92,256)))
BRANCO_VERDE = LinearSegmentedColormap.from_list("BrancoVerde", ["#ffffff","#e8f4ea","#9ed4a9","#2e8b57"])
LARANJA_VERDE = LinearSegmentedColormap.from_list("LaranjaVerde", ["#c2570a","#f3b579","#f7f7f7","#9ed4a9","#1d6b3a"])

def salvar(fig, nome_arquivo, png=True):
    """Grava a figura em figures/capitulo5/ (PDF para o .tex, PNG para conferência)."""
    fig.savefig(FIGDIR / f"{nome_arquivo}.pdf", bbox_inches="tight")
    if png: fig.savefig(FIGDIR / f"{nome_arquivo}.png", bbox_inches="tight", dpi=200)
    print(f"→ figures/capitulo5/{nome_arquivo}.pdf")

def gravar_tabela(df, nome_arquivo):
    """Grava a tabela que sustenta uma figura (proveniência do número publicado)."""
    p = TABDIR / f"{nome_arquivo}.csv"; df.to_csv(p); print(f"→ {p.relative_to(UA)}  {df.shape}")
    return df

vg = lambda v, n=2: f"{v:.{n}f}".replace(".", ",")   # vírgula decimal, para rótulo de figura
print("setup ok ·", len(ASSISTIDOS), "configs assistidas +", len(PISOS), "pisos ·", len(PROBLEMAS), "problemas")

## 0.2 · As famílias de algoritmos

As famílias são as do Capítulo 3 (§3.7): agrupamento hierárquico aglomerativo com ligação média
(UPGMA) sobre a distância de Gower ponderada entre os perfis taxonômicos dos 127 estudos. O corte em
$k=4$ dá as quatro famílias; o corte em $k=7$ desdobra a ameaça evolutiva em três subfamílias pela
origem do número da incerteza. **O grupo de um único algoritmo — o caso isolado do corte — fica fora
de toda agregação por família**, e é reportado sempre em separado.

O código abaixo **reproduz** o agrupamento do notebook 3 a partir de `base127.json` (nada é carregado
de memória) e projeta o resultado sobre as 16 configurações assistidas da bateria.

In [ ]:
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform

EIXOS = ['funcao','motor','regime','surrogate','medicao']
PESOS = {'funcao':0.40,'motor':0.30,'regime':0.10,'surrogate':0.10,'medicao':0.10}   # D1
ATITUDE  = {'OV':'oportunidade','OM':'oportunidade','OL':'oportunidade',
            'AV':'ameaça','AM':'ameaça','AR':'ameaça'}
OBJETO   = {'OV':'valor','AV':'valor','OM':'modelo','AM':'modelo','OL':'próprio-OL','AR':'próprio-AR'}
PARADIGMA= {'EA-Dominância':'EA','EA-Decomposição':'EA','EA-Indicador':'EA',
            'BO-Melhoria-de-Pareto':'BO','BO-Decomposição':'BO','BO-Hipervolume':'BO','BO-Especiais':'BO'}
PRINCIPIO= {'EA-Dominância':'dominância','BO-Melhoria-de-Pareto':'dominância',
            'EA-Decomposição':'decomposição','BO-Decomposição':'decomposição',
            'EA-Indicador':'indicador','BO-Hipervolume':'indicador','BO-Especiais':'especiais'}
TIPO_SUR = {'GP':'regressor','RG':'regressor','NN':'regressor','CL':'classificador'}

base = json.load(open(DISS/'SPEC/bases/base127.json', encoding='utf-8'))
ids  = sorted(base)
caracter = lambda v,e: None if v[e].startswith('Res') else v[e]
dfi = pd.DataFrame({e:[caracter(base[i],e) for i in ids] for e in EIXOS}, index=ids)
dfi['acronimo'] = [base[i]['acronimo'][:24] for i in ids]

def d_ramo_classe(a,b,ramo,classe):
    if a==b: return 0.0
    if ramo[a]==ramo[b]: return 1/3
    if classe[a]==classe[b]: return 2/3
    return 1.0
D_EIXO = {'funcao':      lambda a,b: d_ramo_classe(a,b,ATITUDE,OBJETO),
          'motor':       lambda a,b: d_ramo_classe(a,b,PARADIGMA,PRINCIPIO),
          'regime':      lambda a,b: 0.0 if a==b else 1.0,
          'surrogate':   lambda a,b: 0.0 if a==b else (0.5 if TIPO_SUR[a]==TIPO_SUR[b]=='regressor' else 1.0),
          'medicao':     lambda a,b: 0.0 if a==b else 1.0}

def gower(i,j):
    num=den=0.0
    for e,w in PESOS.items():
        a,b = dfi.at[i,e], dfi.at[j,e]
        if pd.isna(a) or pd.isna(b): continue
        num += w*D_EIXO[e](a,b); den += w
    return num/den

nomeados = dfi.index[dfi['funcao'].notna()].tolist()          # 121: os 6 de resíduo de função saem (D3)
M = np.zeros((len(nomeados),len(nomeados)))
for p,i in enumerate(nomeados):
    for q in range(p+1,len(nomeados)):
        M[p,q] = M[q,p] = gower(i,nomeados[q])
Z = linkage(squareform(M, checks=False), method='average')

def cortar(Z,k):
    n = Z.shape[0]+1; pai = list(range(2*n-1))
    def raiz(x):
        while pai[x]!=x: pai[x]=pai[pai[x]]; x=pai[x]
        return x
    for m in range(n-k):
        pai[raiz(int(Z[m,0]))] = pai[raiz(int(Z[m,1]))] = n+m
    r = [raiz(i) for i in range(n)]; ordem = {x:j for j,x in enumerate(dict.fromkeys(r))}
    rot = np.array([ordem[x] for x in r])
    tam = pd.Series(rot).value_counts().index.tolist()
    return np.array([tam.index(x)+1 for x in rot])

fam = pd.DataFrame({'id':nomeados,'fam_k4':cortar(Z,4),'fam_k7':cortar(Z,7)}).set_index('id')

# Nomes oficiais (§3.7): k=4 nomeia as famílias; k=7 nomeia as subfamílias da ameaça evolutiva.
NOME_FAM = {1:'oportunidade bayesiana', 2:'ameaça evolutiva', 3:'oportunidade evolutiva', 4:'ameaça bayesiana'}
SIGLA_FAM= {1:'OB', 2:'AE', 3:'OE', 4:'AB'}
SUB_K7   = {3:'AE analítica', 4:'AE por erro empírico', 5:'AE por discordância'}
SIGLA_SUB= {3:'AEa', 4:'AEe', 5:'AEd'}
# O caso isolado do corte em k=7 (grupo de 1 algoritmo) fica FORA de toda agregação.
ISOLADO_K7 = fam['fam_k7'].value_counts().pipe(lambda s: s[s==1].index.tolist())
IDS_ISOLADOS = fam.index[fam['fam_k7'].isin(ISOLADO_K7)].tolist()
print('grupo(s) de 1 algoritmo em k=7:', ISOLADO_K7, '→', [dfi.at[i,'acronimo'] for i in IDS_ISOLADOS])

# ── Projeção sobre a bateria ─────────────────────────────────────────────────
linhas = []
for a in ASSISTIDOS:
    i = CONFIG2ID[a]
    k4, k7 = int(fam.at[i,'fam_k4']), int(fam.at[i,'fam_k7'])
    isolado = i in IDS_ISOLADOS
    linhas.append({'alg':a, 'id':i, 'papel':'assistido',
                   'familia':  'isolado' if isolado else SIGLA_FAM[k4],
                   'familia_nome': 'caso isolado' if isolado else NOME_FAM[k4],
                   'grupo':    'isolado' if isolado else SIGLA_SUB.get(k7, SIGLA_FAM[k4]),
                   'grupo_nome':'caso isolado' if isolado else SUB_K7.get(k7, NOME_FAM[k4]),
                   'motor': dfi.at[i,'motor'], 'surrogate': dfi.at[i,'surrogate'],
                   'funcao': dfi.at[i,'funcao'], 'medicao': dfi.at[i,'medicao'],
                   'regime': dfi.at[i,'regime'], 'paradigma_motor': PARADIGMA[dfi.at[i,'motor']]})
for a in PISOS:
    linhas.append({'alg':a,'id':None,'papel':'piso','familia':'piso','familia_nome':'piso sem surrogate',
                   'grupo':'piso','grupo_nome':'piso sem surrogate','motor':None,'surrogate':None,
                   'funcao':None,'medicao':None,'regime':'offline' if a=='moead_media' else 'online','paradigma_motor':'EA'})
df_alg = pd.DataFrame(linhas).set_index('alg')
df_alg['nome'] = [nome(a) for a in df_alg.index]

ORDEM_GRUPO = ['OB','OE','AB','AEa','AEe','AEd','isolado','piso']
COR_GRUPO = {'OB':'#1f77b4','OE':'#2ca02c','AB':'#9467bd','AEa':'#d62728',
             'AEe':'#ff7f0e','AEd':'#8c564b','isolado':'#7f7f7f','piso':'#bdbdbd'}
GRUPO_ALG   = df_alg['grupo'].to_dict()
FAMILIA_ALG = df_alg['familia'].to_dict()

display(df_alg[['nome','papel','familia','grupo','motor','surrogate','funcao','medicao','regime']])
gravar_tabela(df_alg, 'tab_50_plantel_familias')

## 0.3 · As características dos problemas

A matriz de seleção do Capítulo 4 (Figura 4.2 e Apêndice dos problemas): cada problema ocupa uma ou
mais **raias**, cada raia pertence a um **paradigma** de dificuldade, e dentro da raia o problema
ocupa um **degrau** da escada. Doze problemas aparecem em duas raias — é por construção, porque
estressam mais de uma propriedade.

⚠ **Ressalva declarada (D98).** A matriz vem de `caracteristicas_v3.csv`, cujo estatuto de
re-derivação segue pendente no repositório experimental. Ela é usada aqui como está, e é a mesma que
gerou a Figura 4.2 do Capítulo 4 — qualquer revisão dela reflete nos dois lugares.

In [ ]:
carac = pd.read_csv(CACHE/'caracteristicas_v3.csv')
carac['problema'] = carac['problema'].replace({f'F{n}': f'BBOB_F{n}' for n in [1,5,17,22,37,49,55]})

# O paradigma 'maldição da dimensionalidade' tem raia única e escada = o próprio D:
# fica fora da matriz de raias (é a leva 3) e entra aqui só como tabela de dimensão.
DIMENSAO = (carac[carac.grupo_nome=='maldição da dimensionalidade']
              .set_index('problema')[['suite','M','D','orcamento']])
if DIMENSAO.empty:
    DIMENSAO = carac.drop_duplicates('problema').set_index('problema')[['suite','M','D','orcamento']]

EXCLUI_GRUPO = {'maldição da dimensionalidade','avaliação cross (nº de grupos acumulados)','BBOB'}
matriz = carac[~carac.grupo_nome.isin(EXCLUI_GRUPO)].copy()
matriz['paradigma'] = matriz['grupo_nome']
matriz['raia_nome'] = matriz['trilha'].str.split(' — ').str[0].str.strip()
# Há raias homônimas em paradigmas diferentes ("única"): a chave qualifica pelo paradigma.
matriz['raia'] = matriz['paradigma'] + ' · ' + matriz['raia_nome']
matriz['degrau_ord'] = matriz['degrau'].replace({9:99})     # 9 = chefe de fase; vai para o fim
matriz['eh_chefe'] = matriz['degrau'] == 9

PARADIGMAS = matriz.groupby('paradigma')['problema'].nunique().sort_values(ascending=False)
print('paradigmas da matriz:'); print(PARADIGMAS.to_string())
print('\nproblemas em mais de uma raia:',
      matriz.groupby('problema').size().pipe(lambda s: s[s>1]).index.tolist())

PROB_PARADIGMA = matriz.groupby('problema')['paradigma'].apply(list).to_dict()
PROB_RAIA      = matriz.groupby('problema')['raia'].apply(list).to_dict()
RAIA_NOME      = matriz.drop_duplicates('raia').set_index('raia')['raia_nome'].to_dict()
DIM = DIMENSAO['D'].to_dict(); OBJ = DIMENSAO['M'].to_dict(); ORC = DIMENSAO['orcamento'].to_dict()
display(matriz[['paradigma','raia_nome','degrau','degrau_rotulo','problema','suite','M','D']].head(12))
gravar_tabela(matriz, 'tab_51_matriz_problemas')

## 0.4 · As métricas de endpoint

**IGD+** é a métrica primária (D70) e o **HV** a secundária (D69/D92). As duas se calculam sobre os
objetivos normalizados pela régua congelada de cada problema — `ideal` e `nadir` do front verdadeiro,
fixados **antes** de qualquer resultado —, com ref-point $1{,}1$ em cada coordenada. O conjunto
avaliado é o não dominado das avaliações **verdadeiras**: camada ① no regime *online*, camada ⑦ nos
*offline*, onde o conjunto final é reavaliado na função verdadeira.

Duas colunas derivadas tornam os números legíveis e comparáveis **dentro** do problema:

* **`hv_pct`** — o hipervolume da execução como fração do hipervolume do **front verdadeiro** sob a
  mesma régua: $0\%$ = nada dominado, $100\%$ = a frente perfeita. É a resposta à pergunta "quanto da
  solução perfeita este algoritmo alcançou?".
* **`igdp_pct`** — o IGD+ reescalado pelo pior IGD+ observado no problema:
  $100\% \cdot (1 - \mathrm{IGD}^+ / \mathrm{IGD}^+_{\text{pior}})$, com $100\%$ = na frente.

O `cache_endpoint.parquet` é o cômputo oficial da camada `src/metrics.py`. A célula seguinte
**reconfere** uma amostra recalculando do zero a partir dos artefatos brutos.

In [ ]:
end = pd.read_parquet(CACHE/'cache_endpoint.parquet')
end = end[end.alg.isin(ALGORITMOS)].copy()
end['semente'] = end['semente'].astype(int)   # chega como texto em parte dos caches                    # tira c311 (fora do roster) e sub-estudos
end['ok'] = end['status'].eq('ok') & end['igd_plus'].notna() | (end['problema'].isin(SEM_FRONT) & end['hv'].notna())
print('células:', len(end), '| ok:', int(end.ok.sum()), '| falhas:', int((~end.ok).sum()))
print('motivos de falha:', end.loc[~end.ok,'motivo'].fillna('(sem motivo)').value_counts().to_dict())

# HV do FRONT VERDADEIRO por problema, sob a MESMA régua — o denominador do hv_pct.
import sys; sys.path.insert(0, str(UA))
from src import metrics as MET
frentes = pd.read_parquet(CACHE/'frentes_verdadeiras.parquet')

def hv_front_verdadeiro(p):
    if p in SEM_FRONT: return np.nan
    F = frentes.loc[frentes.problema==p, ['f0','f1','f2']].dropna(axis=1).to_numpy(float)
    ideal, nadir = MET.reference_bounds(p)
    return MET.hv(MET.nondominated_front(MET.normalize(F, ideal, nadir)))

HV_MAX = {p: hv_front_verdadeiro(p) for p in PROBLEMAS}
print('\nHV do front verdadeiro (régua S.5, ref 1,1):')
print(pd.Series(HV_MAX).round(4).to_string())

In [ ]:
# ── Caminho dos artefatos no corpus entregue ────────────────────────────────
# O corpus está organizado por <config>/<problema>/<semente>/, e não no layout
# data/experiments/ da frota de execução — por isso o caminho é montado aqui.
def caminho(alg, problema, semente, camada):
    exp = OFFLINES.get(alg, 'main')
    return CORPUS/alg/problema/str(semente)/f'exp_{exp}_{alg}_{problema}_{semente}__{camada}.parquet'

def endpoint_F(alg, problema, semente):
    """Objetivos do conjunto avaliado: camada ⑦ no offline, camada ① no online."""
    p = caminho(alg, problema, semente, 'final' if alg in OFFLINES else 'real')
    if not p.exists():
        p = caminho(alg, problema, semente, 'real')
        if not p.exists(): return None
    return pd.read_parquet(p).filter(regex=r'^f\d+$').to_numpy(float)

from functools import lru_cache
@lru_cache(maxsize=None)
def ref_set(problema):
    """Conjunto de referência do problema — caro de gerar, calculado uma vez só."""
    return None if problema in SEM_FRONT else MET.reference_set(problema)

def metricas_celula(alg, problema, semente):
    """IGD+ e HV oficiais (src/metrics.py) de uma célula, recalculados do bruto."""
    F = endpoint_F(alg, problema, semente)
    if F is None or len(F) == 0: return None
    ideal, nadir = MET.reference_bounds(problema)
    A = MET.nondominated_front(MET.normalize(F, ideal, nadir))
    R = ref_set(problema)
    return {'igd_plus': MET.igd_plus(A, R) if R is not None else np.nan,
            'hv': MET.hv(A), 'n_nd': len(A), 'n_aval': len(F)}

# ── Reconferência (§3 da lei: número nasce de execução na sessão corrente) ──
amostra = end[end.ok].sample(24, random_state=7)[['exp','alg','problema','semente','igd_plus','hv']]
conf = []
for _, r in amostra.iterrows():
    m = metricas_celula(r.alg, r.problema, int(r.semente)) or {}
    conf.append({'alg':r.alg, 'problema':r.problema, 'semente':int(r.semente),
                 'igdp_cache':r.igd_plus, 'igdp_recalc':m.get('igd_plus'),
                 'hv_cache':r.hv, 'hv_recalc':m.get('hv')})
conf = pd.DataFrame(conf)
conf['d_igdp'] = (conf['igdp_cache']-conf['igdp_recalc']).abs()
conf['d_hv']   = (conf['hv_cache']-conf['hv_recalc']).abs()
print(f"reconferência de {len(conf)} células · maior divergência IGD+ = {conf['d_igdp'].max():.2e}"
      f" · HV = {conf['d_hv'].max():.2e}")
display(conf.head(6).round(6))

In [ ]:
# ── Recuperação: as células que o cômputo original perdeu por caminho de arquivo ──
# O resultado é cacheado em disco: reexecutar o notebook não repete o recálculo.
CACHE_REC = CACHE/'cap5_endpoint_recuperado.parquet'
if CACHE_REC.exists():
    end = pd.read_parquet(CACHE_REC)
    print(f'cache de recuperação lido · células válidas: {int(end.ok.sum())}')
else:
    perdidas = end[(~end.ok) & end.motivo.fillna('').str.startswith('erro_metrica:FileNotFound')]
    print('células a recuperar:', len(perdidas))
    linhas_rec = []
    for _, r in perdidas.iterrows():
        m = metricas_celula(r.alg, r.problema, int(r.semente))
        if m is None: continue
        linhas_rec.append({'exp':r.exp, 'alg':r.alg, 'problema':r.problema, 'semente':r.semente,
                           **m, 'status':'ok', 'motivo':'recuperada nesta sessão',
                           'igd':np.nan, 'gd':np.nan, 'spacing':np.nan, 'ok':True})
    if linhas_rec:
        rec = pd.DataFrame(linhas_rec)
        end = pd.concat([end.drop(index=perdidas.index), rec], ignore_index=True)
        print(f'recuperadas: {len(rec)} · células válidas agora: {int(end.ok.sum())}')
    else:
        print('nada a recuperar')
    end.to_parquet(CACHE_REC)


In [ ]:
# ── A tabela-mestra das análises: uma linha por célula (config × problema × semente) ──
df = end[end.ok].copy()
df['regime'] = np.where(df.exp=='off','offline','online')
df = df.join(df_alg[['nome','papel','familia','grupo','motor','surrogate','funcao','medicao','paradigma_motor']], on='alg')
df['D'] = df['problema'].map(DIM); df['M'] = df['problema'].map(OBJ); df['orcamento'] = df['problema'].map(ORC)
df['suite'] = df['problema'].map(DIMENSAO['suite'])
df['tipo_problema'] = np.where(df.problema.isin(PROBLEMAS_REAIS),'real','sintético')

# hv_pct: fração do HV do front verdadeiro (0-100%), clipada em [0,100]
df['hv_pct'] = 100*df['hv']/df['problema'].map(HV_MAX)
df['hv_pct'] = df['hv_pct'].clip(0,100)
# igdp_pct: 100% = na frente; 0% = o pior observado no problema
pior = df.groupby('problema')['igd_plus'].transform(lambda s: s.quantile(0.99))
df['igdp_pct'] = (100*(1 - df['igd_plus']/pior)).clip(0,100)

# rank dentro de cada problema × semente (1 = melhor), por métrica
for m, asc in [('igd_plus',True), ('hv',False)]:
    df[f'rank_{m}'] = df.groupby(['problema','semente','regime'])[m].rank(ascending=asc, method='average')

print(df.shape, '| células por regime:', df.regime.value_counts().to_dict())
display(df.head(4)[['alg','nome','grupo','problema','semente','regime','igd_plus','hv','hv_pct','igdp_pct','rank_igd_plus']])
gravar_tabela(df, 'tab_52_celulas_metricas')

## 0.5 · Completude: o que pousou, e o que não pousou

O placar honesto antes de qualquer comparação. Uma célula é *válida* quando produziu métrica sob a
régua oficial; as demais entram em categorias **nomeadas** — nenhuma some em silêncio.

In [ ]:
cov = (end.assign(valida=end.ok)
          .pivot_table(index='alg', columns='problema', values='valida', aggfunc='sum')
          .reindex(index=ALGORITMOS, columns=PROBLEMAS).fillna(0).astype(int))

fig, (ax, axb) = plt.subplots(1, 2, figsize=(15, 6.2), gridspec_kw={'width_ratios':[3.1,1]})
sns.heatmap(cov, ax=ax, cmap=BRANCO_VERDE, vmin=0, vmax=30, annot=True, fmt='d',
            annot_kws={'fontsize':6.5}, linewidths=0.4, linecolor='#eeeeee',
            cbar_kws={'label':'sementes válidas (de 30)', 'shrink':0.7})
ax.axhline(len(ASSISTIDOS), color='k', lw=1.4)
ax.axvline(len(PROBLEMAS_SINT), color='k', lw=1.4, ls='--')
ax.set_yticklabels([nome(a) for a in cov.index], rotation=0, fontsize=8)
ax.set_xticklabels(cov.columns, rotation=55, ha='right', fontsize=7.5)
ax.set_xlabel(''); ax.set_ylabel(''); ax.set_title('(a) sementes válidas por configuração × problema', fontsize=11)

mot = (end.loc[~end.ok,'motivo'].fillna('não coletada').value_counts())
axb.barh(range(len(mot)), mot.values, color='#c2570a', alpha=0.85)
axb.set_yticks(range(len(mot))); axb.set_yticklabels([m[:26] for m in mot.index], fontsize=8)
axb.invert_yaxis(); axb.set_xlabel('células'); axb.set_title('(b) por que faltam', fontsize=11)
for i, v in enumerate(mot.values): axb.text(v, i, f' {v}', va='center', fontsize=8)
axb.spines[['top','right']].set_visible(False)
plt.tight_layout(); salvar(fig, 'fig511_completude'); plt.show()

print('total previsto:', len(ALGORITMOS)*len(PROBLEMAS)*30, '| válidas:', int(cov.values.sum()),
      f'({100*cov.values.sum()/(len(ALGORITMOS)*len(PROBLEMAS)*30):.1f}%)')
gravar_tabela(cov, 'tab_53_completude')

---
# Parte 1 · Característica do problema × característica do algoritmo

> *"que característica do algoritmo explica o resultado, diante de que característica do problema?"*
> — a pergunta-guia declarada na Seção 4.1.

Esta é a leva para a qual o desenho experimental foi construído. Três passos: **(1)** a tabela
agrupada família × paradigma, com a distribuição por trás de cada célula; **(2)** as escadas de
dificuldade, que dizem em que **degrau** cada família deixa de funcionar; **(3)** os vencedores por
raia, com o piso casado como régua.

### 1.1 · Desempenho por família de algoritmo × paradigma de problema

Cada célula problema-semente entra em todas as raias do problema — e, portanto, em todos os
paradigmas que ele estressa. A leitura é por **rank dentro do problema-semente**, a única agregação
lícita entre problemas de orçamento diferente, e por **HV alcançado** (`hv_pct`), que é interpretável
em si. Os pisos entram como régua na mesma tabela.

In [ ]:
# Explode as células pelos paradigmas do problema (12 problemas ocupam duas raias)
par_long = (df[df.tipo_problema=='sintético']
              .merge(matriz[['problema','paradigma','raia','degrau','degrau_rotulo','eh_chefe']],
                     on='problema', how='inner'))
print('linhas célula × paradigma:', len(par_long), '| problemas cobertos:', par_long.problema.nunique())

ORD_PAR = (matriz.groupby('paradigma')['problema'].nunique().sort_values(ascending=False).index.tolist())
GRUPOS_VIS = [g for g in ORDEM_GRUPO if g in set(par_long['grupo'])]

def painel_grupo_paradigma(valor, titulo, cmap, fmt='{:.1f}', invert=False, ax=None):
    t = (par_long[par_long.regime=='online']
           .groupby(['grupo','paradigma'])[valor].median().unstack('paradigma')
           .reindex(index=GRUPOS_VIS, columns=ORD_PAR))
    fig_, ax = (plt.subplots(figsize=(8,4.2)) if ax is None else (ax.figure, ax))
    sns.heatmap(t, ax=ax, cmap=cmap, annot=t.map(lambda v: fmt.format(v) if pd.notna(v) else ''),
                fmt='', linewidths=0.6, linecolor='white', cbar=False,
                annot_kws={'fontsize':9})
    ax.set_xticklabels([p.replace(' / ','/\n').replace(' (côncavo, desconexo)','') for p in t.columns],
                       rotation=20, ha='right', fontsize=8)
    ax.set_yticklabels(t.index, rotation=0, fontsize=9.5)
    ax.set_xlabel(''); ax.set_ylabel(''); ax.set_title(titulo, fontsize=10.5)
    return t

fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.6))
t_rank = painel_grupo_paradigma('rank_igd_plus', '(a) rank médio de IGD+ (menor = melhor)',
                                GREENS.reversed(), '{:.1f}', ax=axes[0])
t_hv   = painel_grupo_paradigma('hv_pct', '(b) hipervolume alcançado (% do front verdadeiro)',
                                BRANCO_VERDE, '{:.0f}', ax=axes[1])
plt.tight_layout(); salvar(fig, 'fig521_familia_paradigma'); plt.show()
display(t_rank.round(2)); gravar_tabela(t_rank, 'tab_54_rank_grupo_paradigma')
gravar_tabela(t_hv, 'tab_55_hvpct_grupo_paradigma')

### 1.2 · A distribuição por trás de cada célula

A mediana da tabela esconde a dispersão — e a dispersão é resultado. Os painéis abaixo abrem, por
paradigma, a distribuição completa do hipervolume alcançado, com a banda dos pisos marcada: a faixa
que um evolutivo puro entrega sob o mesmo orçamento. Cair dentro da banda significa não comprar nada
com o modelo.

In [ ]:
banda = (par_long[(par_long.papel=='piso') & (par_long.regime=='online')]
           .groupby('paradigma')['hv_pct'].quantile([0.25,0.75]).unstack())

fig, axes = plt.subplots(2, 3, figsize=(15.5, 7.4), sharey=True)
for ax, p in zip(axes.ravel(), ORD_PAR):
    sub = par_long[(par_long.paradigma==p) & (par_long.regime=='online') & (par_long.grupo!='isolado')]
    ordem = [g for g in GRUPOS_VIS if g in set(sub['grupo'])]
    ordem = [g for g in ordem if (sub['grupo']==g).any()]
    sns.boxplot(data=sub, x='grupo', y='hv_pct', order=ordem, hue='grupo', hue_order=ordem,
                palette={g: COR_GRUPO[g] for g in ordem}, legend=False,
                ax=ax, width=0.66, fliersize=0.7, linewidth=0.8)
    if p in banda.index:
        ax.axhspan(banda.loc[p,0.25], banda.loc[p,0.75], color='#bdbdbd', alpha=0.30, zorder=0)
    ax.set_title(f'{p}  ({sub.problema.nunique()} problemas)', fontsize=10)
    ax.set_xlabel(''); ax.set_ylabel('HV alcançado (%)' if any(ax is a for a in axes[:,0]) else '')
    ax.tick_params(axis='x', labelsize=9); ax.spines[['top','right']].set_visible(False)
for ax in axes.ravel()[len(ORD_PAR):]: ax.set_visible(False)
axes.ravel()[0].legend(handles=[Patch(facecolor='#bdbdbd', alpha=0.4, label='faixa interquartil dos pisos')],
                       fontsize=8, frameon=False, loc='lower left')
plt.tight_layout(); salvar(fig, 'fig522_distribuicoes_paradigma'); plt.show()

### 1.3 · As escadas de dificuldade: em que degrau cada família quebra

Cada raia é uma escada de severidade crescente da mesma propriedade. A pergunta que o desenho
autoriza não é *se* uma família falha, mas **a partir de que degrau** — e a resposta é a inclinação
da curva. O **degrau de falha** de uma família numa raia é o primeiro em que ela deixa de superar a
mediana dos pisos naquele degrau.

In [ ]:
esc = (par_long[(par_long.regime=='online') & (par_long.grupo!='isolado')]
         .groupby(['raia','paradigma','degrau','degrau_rotulo','grupo'])['hv_pct'].median().reset_index())
piso_deg = esc[esc.grupo=='piso'].groupby(['raia','degrau'])['hv_pct'].median()
esc['vs_piso'] = esc['hv_pct'] - esc.set_index(['raia','degrau']).index.map(piso_deg)

RAIAS = [r for r in matriz.sort_values(['paradigma','raia']).raia.unique() if (esc.raia==r).sum() > 0]
ncol = 3; nrow = int(np.ceil(len(RAIAS)/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.2*ncol, 3.1*nrow), sharey=True)
for ax, r in zip(axes.ravel(), RAIAS):
    sub = esc[esc.raia==r].sort_values('degrau')
    rot = sub.drop_duplicates('degrau').sort_values('degrau')
    for g in [x for x in GRUPOS_VIS if x in set(sub['grupo'])]:
        s = sub[sub.grupo==g].sort_values('degrau')
        ax.plot(range(len(s)), s['hv_pct'], 'o-', ms=4.5, lw=1.6 if g!='piso' else 2.2,
                color=COR_GRUPO[g], ls='-' if g!='piso' else '--', label=g, zorder=3 if g!='piso' else 2)
    ax.set_xticks(range(len(rot))); ax.set_xticklabels(rot['degrau_rotulo'], fontsize=8)
    ax.set_title(RAIA_NOME.get(r, r)[:46], fontsize=9.5); ax.grid(axis='y', color='#eeeeee', lw=0.7)
    ax.spines[['top','right']].set_visible(False)
    if axes.ndim == 1 or any(ax is a for a in axes[:,0]): ax.set_ylabel('HV alcançado (%)')
for ax in axes.ravel()[len(RAIAS):]: ax.set_visible(False)
h, l = axes.ravel()[0].get_legend_handles_labels()
fig.legend(h, l, ncol=len(l), fontsize=9, frameon=False, loc='lower center', bbox_to_anchor=(0.5,-0.02))
plt.tight_layout(); salvar(fig, 'fig523_escadas'); plt.show()
gravar_tabela(esc, 'tab_56_escadas')

In [ ]:
# O degrau de falha: o primeiro em que a família deixa de superar o piso na raia
falhas = []
for (r, g), s in esc[esc.grupo!='piso'].groupby(['raia','grupo']):
    s = s.sort_values('degrau')
    perde = s[s['vs_piso'] <= 0]
    falhas.append({'raia':r, 'grupo':g, 'degraus':len(s),
                   'degrau_falha': perde['degrau_rotulo'].iloc[0] if len(perde) else '—',
                   'vs_piso_primeiro': s['vs_piso'].iloc[0], 'vs_piso_chefe': s['vs_piso'].iloc[-1],
                   'queda': s['hv_pct'].iloc[0] - s['hv_pct'].iloc[-1]})
tab_falha = pd.DataFrame(falhas).pivot(index='grupo', columns='raia', values='degrau_falha')
tab_falha = tab_falha.reindex(index=[g for g in GRUPOS_VIS if g not in ('piso',)])
print('Degrau em que cada família deixa de superar o piso ("—" = nunca deixa):')
display(tab_falha)
gravar_tabela(pd.DataFrame(falhas), 'tab_57_degrau_falha')

### 1.4 · Os vencedores: quem ganha onde, e por quanto

A pergunta fecha com o placar **pareado**. Como assistido e piso partem da mesma amostra inicial em
cada par problema-semente, a diferença entre eles mede o surrogate e não a sorte do sorteio. O
confronto é contra o **piso casado ao princípio de seleção** do algoritmo; os três da classe
BO · especiais, sem análogo populacional, são lidos contra a **banda** — o melhor dos quatro pisos em
cada célula, o teste mais duro do estudo.

In [ ]:
def delta_vs_piso(metrica='igd_plus', menor_melhor=True, regimes=('online',)):
    """Diferença pareada (piso − assistido, orientada para 'positivo = o assistido ganha')."""
    d = df[df.regime.isin(regimes)]
    pisos_w = (d[d.papel=='piso'].pivot_table(index=['problema','semente'], columns='alg', values=metrica))
    banda_ = pisos_w[[p for p in PISOS if p in pisos_w and p!='moead_media']].min(axis=1) if menor_melhor \
             else pisos_w[[p for p in PISOS if p in pisos_w and p!='moead_media']].max(axis=1)
    out = []
    for a in [x for x in ASSISTIDOS if x in set(d.alg)]:
        ref = PISO_CASADO[a]
        serie_piso = banda_ if ref=='BANDA' else (pisos_w[ref] if ref in pisos_w else None)
        if serie_piso is None: continue
        s = d[d.alg==a].set_index(['problema','semente'])[metrica]
        j = pd.concat({'alg':s, 'piso':serie_piso}, axis=1).dropna()
        if j.empty: continue
        delta = (j['piso'] - j['alg']) if menor_melhor else (j['alg'] - j['piso'])
        out.append(pd.DataFrame({'alg':a, 'ref':ref, 'delta':delta}).reset_index())
    return pd.concat(out, ignore_index=True)

dv_on  = delta_vs_piso('igd_plus', True, ('online',))
dv_off = delta_vs_piso('igd_plus', True, ('offline',))
dv = pd.concat([dv_on, dv_off], ignore_index=True)
dv['grupo'] = dv['alg'].map(GRUPO_ALG); dv['nome'] = dv['alg'].map(nome)
dv['vence'] = dv['delta'] > 0

placar = (dv[dv.problema.isin(PROBLEMAS_SINT)]
            .groupby(['alg','nome','grupo','ref','problema'])['delta'].median().reset_index()
            .assign(v=lambda t: t.delta>0)
            .groupby(['alg','nome','grupo','ref'])
            .agg(V=('v','sum'), N=('v','size'), delta_mediano=('delta','median')).reset_index())
placar['D'] = placar['N'] - placar['V']
placar = placar.sort_values('V', ascending=False)

fig, ax = plt.subplots(figsize=(9.2, 5.6))
y = np.arange(len(placar))
ax.barh(y, placar['V'], color=[COR_GRUPO[g] for g in placar['grupo']], alpha=0.92, label='vitórias')
ax.barh(y, -placar['D'], color='#c2570a', alpha=0.55, label='derrotas')
ax.axvline(0, color='k', lw=0.9)
ax.set_yticks(y); ax.set_yticklabels([f"{r.nome}  ({r.grupo} · vs {nome(r.ref) if r.ref!='BANDA' else 'banda'})"
                                      for r in placar.itertuples()], fontsize=9)
ax.invert_yaxis(); ax.set_xlabel('problemas sintéticos (mediana pareada de IGD+)')
for i, r in enumerate(placar.itertuples()):
    ax.text(r.V+0.35, i, str(r.V), va='center', fontsize=8.5, color='#1d6b3a')
    ax.text(-r.D-0.35, i, str(r.D), va='center', ha='right', fontsize=8.5, color='#c2570a')
ax.set_title('O surrogate compra alguma coisa? Placar contra o piso casado', fontsize=11)
ax.legend(fontsize=8.5, frameon=False, loc='lower right'); ax.spines[['top','right','left']].set_visible(False)
plt.tight_layout(); salvar(fig, 'fig524_placar_pisos'); plt.show()
display(placar[['nome','grupo','ref','V','D','delta_mediano']])
gravar_tabela(placar, 'tab_58_placar_pisos')

In [ ]:
# Vencedor por raia: o algoritmo e a família de melhor rank médio dentro de cada raia
venc = (par_long[(par_long.regime=='online') & (par_long.papel=='assistido')]
          .groupby(['paradigma','raia','alg','nome','grupo'])['rank_igd_plus'].mean().reset_index()
          .sort_values('rank_igd_plus'))
top_raia = venc.groupby(['paradigma','raia']).head(3).sort_values(['paradigma','raia','rank_igd_plus'])
top_raia['raia'] = top_raia['raia'].map(lambda r: RAIA_NOME.get(r, r))
resumo_raia = (top_raia.groupby(['paradigma','raia'])
                 .apply(lambda g: ' · '.join(f"{r.nome} ({r.grupo}, {vg(r.rank_igd_plus,1)})" for r in g.itertuples()))
                 .rename('pódio (rank médio de IGD+)').reset_index())
display(resumo_raia)
gravar_tabela(resumo_raia, 'tab_59_podio_por_raia')

---
# Parte 2 · Aprofundamento: por que cada família vence onde vence

A Parte 1 disse **quem** ganha em **que** paradigma. Esta parte pergunta **por quê**, dificuldade por
dificuldade, com o instrumento próprio de cada uma. O conjunto não dominado de cada execução —
objetivos *e* variáveis de decisão — vem do cache `nd/`, derivado da camada ① (online) e ⑦ (offline).

In [ ]:
ND = {}
def nd(problema):
    """Conjunto não dominado de cada célula do problema (F e X), do cache nd/."""
    if problema not in ND:
        p = CACHE/'nd'/f'nd_{problema}.parquet'
        d = pd.read_parquet(p) if p.exists() else None
        if d is not None: d['semente'] = d['semente'].astype(int)
        ND[problema] = d
    return ND[problema]

def ps_verdadeiro(problema):
    p = CACHE/'nd'/f'ps_verdadeiro_{problema}.parquet'
    return pd.read_parquet(p) if p.exists() else None

def cols_x(d): return sorted([c for c in d.columns if re.fullmatch(r'x\d+', c)], key=lambda c:int(c[1:]))
def cols_f(d): return sorted([c for c in d.columns if re.fullmatch(r'f\d+', c)], key=lambda c:int(c[1:]))
print('problemas com conjunto não dominado em cache:', sum(1 for p in PROBLEMAS if (CACHE/"nd"/f"nd_{p}.parquet").exists()))

### 2.1 · Multimodalidade no espaço de decisão: os modais foram encontrados?

As métricas do espaço de objetivos — IGD+, HV — são cegas a esta pergunta. Dois algoritmos podem ter
o **mesmo** front e, ainda assim, um ter achado os quatro conjuntos de Pareto do MMF4 e o outro ter
colapsado num só. A suíte MMF existe para isso, e a métrica que a acompanha é o **IGDX**, o IGD
calculado no espaço de **decisão**:

$$\mathrm{IGDX}(A) = \frac{1}{|PS^*|}\sum_{p \in PS^*}\ \min_{a \in A}\ \lVert p - a \rVert_2 ,$$

com $PS^*$ o conjunto de Pareto verdadeiro e $A$ as variáveis de decisão das soluções não dominadas
encontradas. Valor baixo só é possível quando a aproximação **cobre todos os ramos** do $PS^*$: cobrir
um único ramo deixa os pontos dos demais sem vizinho próximo e o IGDX explode.

A segunda medida é direta: a **cobertura de modais**. Cada ponto do $PS^*$ é atribuído ao seu modal
(por agrupamento dos ramos), e conta-se quantos modais receberam ao menos uma solução a distância
menor que o raio de captura — a fração deles é a cobertura.

In [ ]:
from scipy.spatial.distance import cdist
from sklearn.cluster import KMeans

# Número de conjuntos de Pareto globais de cada MMF (declarado no Apêndice dos problemas)
N_MODAIS = {'MMF1': 2, 'MMF4': 4, 'MMF11_L': 2, 'MMF16_20': 2}
MMF = [p for p in N_MODAIS if (CACHE/'nd'/f'ps_verdadeiro_{p}.parquet').exists()]

def igdx_e_cobertura(problema, k_modais):
    PS = ps_verdadeiro(problema); d = nd(problema)
    if PS is None or d is None: return None
    xs = cols_x(PS); P = PS[xs].to_numpy(float)
    lab = KMeans(n_clusters=k_modais, n_init=10, random_state=0).fit_predict(P)
    # raio de captura: 2× a distância mediana entre vizinhos dentro do mesmo modal
    dmin = np.array([np.sort(cdist(P[lab==g], P[lab==g]), axis=1)[:,1].mean() for g in range(k_modais)])
    raio = 2*float(np.median(dmin))
    out = []
    for (alg, sem), g in d.groupby(['alg','semente']):
        A = g[xs].to_numpy(float)
        if len(A)==0: continue
        Dm = cdist(P, A)
        igdx = float(Dm.min(axis=1).mean())
        cob = float(np.mean([ (Dm[lab==m].min(axis=1) < raio).any() for m in range(k_modais) ]))
        out.append({'problema':problema,'alg':alg,'semente':sem,'igdx':igdx,'cobertura_modais':cob,'n_nd':len(A)})
    return pd.DataFrame(out)

mm = pd.concat([x for x in (igdx_e_cobertura(p, N_MODAIS[p]) for p in MMF) if x is not None], ignore_index=True)
mm = mm[mm.alg.isin(ALGORITMOS)]
mm['grupo'] = mm['alg'].map(GRUPO_ALG); mm['nome'] = mm['alg'].map(nome)
mm = mm.merge(df[['alg','problema','semente','igd_plus','hv_pct']], on=['alg','problema','semente'], how='left')
print(mm.groupby('problema')[['igdx','cobertura_modais']].median().round(3).to_string())
gravar_tabela(mm, 'tab_60_igdx_modais')

In [ ]:
res_mm = (mm.groupby(['alg','nome','grupo'])
            .agg(igdx=('igdx','median'), cobertura=('cobertura_modais','mean'),
                 igdp=('igd_plus','median')).reset_index().sort_values('igdx'))

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4), gridspec_kw={'width_ratios':[1.25,1.25,1]})
# (a) IGDX por algoritmo
o = res_mm.sort_values('igdx')
axes[0].barh(range(len(o)), o['igdx'], color=[COR_GRUPO[g] for g in o['grupo']], alpha=0.9)
axes[0].set_yticks(range(len(o))); axes[0].set_yticklabels(o['nome'], fontsize=8.5); axes[0].invert_yaxis()
axes[0].set_xlabel('IGDX mediano nos MMF (menor = cobre mais ramos do PS)')
axes[0].set_title('(a) distância no espaço de decisão', fontsize=10.5)
# (b) cobertura de modais
o2 = res_mm.sort_values('cobertura', ascending=False)
axes[1].barh(range(len(o2)), 100*o2['cobertura'], color=[COR_GRUPO[g] for g in o2['grupo']], alpha=0.9)
axes[1].set_yticks(range(len(o2))); axes[1].set_yticklabels(o2['nome'], fontsize=8.5); axes[1].invert_yaxis()
axes[1].set_xlabel('modais cobertos (%)'); axes[1].set_xlim(0,100)
axes[1].set_title('(b) cobertura dos conjuntos de Pareto', fontsize=10.5)
# (c) o divórcio: bom no espaço de objetivos ≠ bom no de decisão
axes[2].scatter(res_mm['igdp'], res_mm['igdx'], s=60, c=[COR_GRUPO[g] for g in res_mm['grupo']],
                edgecolor='k', lw=0.4, alpha=0.9)
for r in res_mm.itertuples():
    axes[2].annotate(r.nome, (r.igdp, r.igdx), fontsize=7, xytext=(4,3), textcoords='offset points')
rho = res_mm[['igdp','igdx']].corr(method='spearman').iloc[0,1]
axes[2].set_xlabel('IGD+ (espaço de objetivos)'); axes[2].set_ylabel('IGDX (espaço de decisão)')
axes[2].set_title(f'(c) as duas leituras · ρ de Spearman = {vg(rho)}', fontsize=10.5)
for ax in axes: ax.spines[['top','right']].set_visible(False); ax.grid(axis='x', color='#f0f0f0', lw=0.7)
plt.tight_layout(); salvar(fig, 'fig531_multimodalidade_igdx'); plt.show()
display(res_mm.round(3)); gravar_tabela(res_mm, 'tab_61_igdx_por_algoritmo')

### 2.2 · As seis dificuldades, uma a uma

Um instrumento por pergunta, sobre o conjunto não dominado de cada execução:

| Dificuldade | Pergunta | Medida |
|---|---|---|
| ótimos locais | o ótimo verdadeiro foi alcançado? | distância do melhor ponto ao ideal (convergência) |
| enganosidade | a bacia falsa prendeu a busca? | fração das execuções acima do limiar de estagnação |
| regiões planas / heterocedasticidade | o modelo travou no platô? | dispersão entre sementes (instabilidade) |
| frente côncava | prendeu nas extremidades? | massa da distribuição no miolo da frente |
| frente desconexa | todas as ilhas foram achadas? | ilhas cobertas / ilhas existentes |
| multimodalidade | os modais foram achados? | IGDX e cobertura (§2.1) |

In [ ]:
def ilhas_cobertas(problema, k_ilhas):
    """Fração das ilhas da frente verdadeira que receberam ao menos uma solução próxima."""
    d = nd(problema)
    F_true = frentes.loc[frentes.problema==problema].dropna(axis=1)
    fs = cols_f(F_true)
    if d is None or F_true.empty: return None
    P = F_true[fs].to_numpy(float)
    ideal, nadir = MET.reference_bounds(problema)
    Pn = MET.normalize(P, ideal, nadir)
    lab = KMeans(n_clusters=k_ilhas, n_init=10, random_state=0).fit_predict(Pn)
    raio = 2*float(np.median([np.sort(cdist(Pn[lab==g],Pn[lab==g]),axis=1)[:,1].mean() for g in range(k_ilhas)]))
    out=[]
    for (alg,sem), g in d.groupby(['alg','semente']):
        A = MET.normalize(g[cols_f(g)].to_numpy(float), ideal, nadir)
        Dm = cdist(Pn, A)
        out.append({'problema':problema,'alg':alg,'semente':sem,
                    'ilhas':float(np.mean([(Dm[lab==m].min(axis=1)<raio).any() for m in range(k_ilhas)]))})
    return pd.DataFrame(out)

def massa_miolo(problema):
    """Fração das soluções não dominadas no miolo da frente (fora dos 20% extremos de cada objetivo)."""
    d = nd(problema)
    if d is None: return None
    ideal, nadir = MET.reference_bounds(problema)
    out=[]
    for (alg,sem), g in d.groupby(['alg','semente']):
        A = MET.normalize(g[cols_f(g)].to_numpy(float), ideal, nadir)
        if len(A)==0: continue
        dentro = ((A>0.2)&(A<0.8)).all(axis=1)
        out.append({'problema':problema,'alg':alg,'semente':sem,'miolo':float(dentro.mean())})
    return pd.DataFrame(out)

N_ILHAS = {'ZDT3':5, 'DTLZ7':4, 'WFG2':4}
ilhas = pd.concat([x for x in (ilhas_cobertas(p,k) for p,k in N_ILHAS.items()) if x is not None], ignore_index=True)
CONCAVOS = ['MMF4','DTLZ2','ZDT6','DTLZ4','DTLZ3']
miolo = pd.concat([x for x in (massa_miolo(p) for p in CONCAVOS) if x is not None], ignore_index=True)
for t in (ilhas, miolo):
    t['grupo'] = t['alg'].map(GRUPO_ALG); t['nome'] = t['alg'].map(nome)
print('ilhas:', ilhas.groupby('problema')['ilhas'].median().round(2).to_dict())
print('miolo:', miolo.groupby('problema')['miolo'].median().round(2).to_dict())

In [ ]:
# Painel das seis dificuldades — uma métrica por painel, agregada por família
LOCAIS   = ['ZDT4','DTLZ1','DTLZ3','WFG4']
ENGANO   = ['WFG5','WFG9']
PLANAS   = ['WFG1']
HETERO   = ['ZDT6','DTLZ4','BBOB_F22']

def por_grupo(tab, col, probs=None):
    t = tab if probs is None else tab[tab.problema.isin(probs)]
    return (t[t.grupo!='isolado'].groupby('grupo')[col].median()
              .reindex([g for g in ORDEM_GRUPO if g in set(t['grupo'])]).dropna())

paineis = [
    ('ótimos locais\n(convergência: HV alcançado)', por_grupo(df[df.regime=='online'], 'hv_pct', LOCAIS), '%'),
    ('enganosidade\n(HV alcançado)',                por_grupo(df[df.regime=='online'], 'hv_pct', ENGANO), '%'),
    ('regiões planas\n(dispersão entre sementes)',  df[(df.regime=='online')&(df.problema.isin(PLANAS))&(df.grupo!='isolado')]
                                                      .groupby(['grupo'])['hv_pct'].std()
                                                      .reindex([g for g in ORDEM_GRUPO if g in set(df['grupo'])]).dropna(), 'dp'),
    ('heterocedasticidade\n(HV alcançado)',         por_grupo(df[df.regime=='online'], 'hv_pct', HETERO), '%'),
    ('frente côncava\n(massa no miolo)',            100*por_grupo(miolo, 'miolo'), '%'),
    ('frente desconexa\n(ilhas cobertas)',          100*por_grupo(ilhas, 'ilhas'), '%'),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, (titulo, serie, un) in zip(axes.ravel(), paineis):
    cores = [COR_GRUPO[g] for g in serie.index]
    ax.bar(range(len(serie)), serie.values, color=cores, alpha=0.92, width=0.68)
    ax.set_xticks(range(len(serie))); ax.set_xticklabels(serie.index, fontsize=9)
    ax.set_title(titulo, fontsize=10)
    for i, v in enumerate(serie.values): ax.text(i, v, f' {vg(v,1)}', ha='center', va='bottom', fontsize=8)
    ax.spines[['top','right']].set_visible(False); ax.grid(axis='y', color='#f0f0f0', lw=0.7)
    ax.set_ylim(0, max(serie.values)*1.18)
plt.tight_layout(); salvar(fig, 'fig532_seis_dificuldades'); plt.show()
gravar_tabela(pd.DataFrame({t: s for t, s, _ in paineis}), 'tab_62_seis_dificuldades')

### 2.3 · As fronteiras: o que os melhores de fato encontraram

Pela convenção do corpus, a execução mostrada é a de **IGD+ mediano** — nunca a melhor. Seis
geometrias difíceis, uma por painel, com a frente verdadeira ao fundo.

In [ ]:
VITRINE = ['ZDT3','DTLZ7','MMF4','WFG9','ZDT4','BBOB_F55']
TOP = 4

def execucao_mediana(alg, problema):
    s = df[(df.alg==alg)&(df.problema==problema)]
    if s.empty: return None
    return int(s.iloc[(s['igd_plus'] - s['igd_plus'].median()).abs().argsort().iloc[0]]['semente'])

fig, axes = plt.subplots(2, 3, figsize=(15, 8.4))
for ax, p in zip(axes.ravel(), VITRINE):
    Ft = frentes.loc[frentes.problema==p].dropna(axis=1); fs = cols_f(Ft)
    if len(fs) == 2:
        ax.scatter(Ft[fs[0]], Ft[fs[1]], s=3, c='#d9d9d9', label='frente verdadeira', zorder=1)
    melhores = (df[(df.problema==p)&(df.papel=='assistido')&(df.regime=='online')]
                  .groupby('alg')['igd_plus'].median().nsmallest(TOP).index.tolist())
    d = nd(p)
    for a in melhores:
        sem = execucao_mediana(a, p)
        g = d[(d.alg==a)&(d.semente==sem)] if d is not None else None
        if g is None or g.empty: continue
        gf = cols_f(g)
        if len(gf) == 2:
            ax.scatter(g[gf[0]], g[gf[1]], s=22, color=CORES.get(a), edgecolor='k', lw=0.3,
                       alpha=0.9, label=nome(a), zorder=3)
    ax.set_title(f'{p}  ·  {matriz.loc[matriz.problema==p,"raia"].iloc[0][:34] if (matriz.problema==p).any() else ""}', fontsize=10)
    ax.set_xlabel('$f_1$'); ax.set_ylabel('$f_2$')
    ax.legend(fontsize=7, frameon=False, markerscale=0.9)
    ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); salvar(fig, 'fig533_fronteiras'); plt.show()

### 2.4 · Contrastes: mesma maquinaria, incerteza diferente

O paradigma narrativo do Capítulo 3 levado ao laboratório. Cada par compartilha o essencial da
maquinaria e difere no tratamento da incerteza — de modo que a diferença medida **isola** esse
tratamento. O par mais limpo é a ablação exata do regime *offline*: Prob-MOEA/D contra MOEA/D-média,
idênticos a menos do uso do desvio-padrão do modelo.

In [ ]:
def contraste(a, b, rotulo, regime='online'):
    d = df[df.regime==regime]
    j = (d[d.alg==a].set_index(['problema','semente'])['igd_plus']
           .to_frame('a').join(d[d.alg==b].set_index(['problema','semente'])['igd_plus'].rename('b'), how='inner'))
    if j.empty: return None
    j['delta'] = j['b'] - j['a']          # positivo = 'a' melhor
    r = j.groupby('problema')['delta'].median().rename('delta')
    return pd.DataFrame({'par':rotulo,'a':a,'b':b,'problema':r.index,'delta':r.values})

PARES = [('b5m','moead_media','ablação exata do σ\nProb-MOEA/D × MOEA/D-média','offline'),
         ('c262','smsemoa','espelho do hipervolume\nqNEHVI × SMS-EMOA','online'),
         ('c217','b4','mesma classe, gerações distintas\nPC-SAEA × CSEA','online'),
         ('e7','b3','rede × processo gaussiano\nEDN-ARMOEA × K-RVEA','online'),
         ('c141','e74','discordância × distância\nMMRAEA × CLMEA','online')]
ctr = pd.concat([x for x in (contraste(a,b,r,g) for a,b,r,g in PARES) if x is not None], ignore_index=True)

fig, axes = plt.subplots(1, len(PARES), figsize=(3.05*len(PARES), 5.6), sharey=False)
for ax, (a,b,rot,_) in zip(np.atleast_1d(axes), PARES):
    s = ctr[ctr.par==rot].sort_values('delta')
    if s.empty: ax.set_visible(False); continue
    cor = ['#1d6b3a' if v>0 else '#c2570a' for v in s['delta']]
    ax.barh(range(len(s)), s['delta'], color=cor, alpha=0.9)
    ax.set_yticks(range(len(s))); ax.set_yticklabels(s['problema'], fontsize=7)
    ax.axvline(0, color='k', lw=0.9)
    v, d_ = int((s.delta>0).sum()), int((s.delta<=0).sum())
    ax.set_title(f'{rot}\n{v} × {d_}', fontsize=9)
    ax.set_xlabel('Δ IGD+ pareado'); ax.spines[['top','right']].set_visible(False)
    ax.tick_params(axis='x', labelsize=7)
plt.tight_layout(); salvar(fig, 'fig534_contrastes'); plt.show()
display(ctr.groupby('par')['delta'].agg(vitorias=lambda s:(s>0).sum(), problemas='size', delta_mediano='median'))
gravar_tabela(ctr, 'tab_63_contrastes')

---
# Parte 3 · A maldição da dimensionalidade e o custo

O quinto paradigma do desenho tem raia única e escada própria: a dimensão $D$ do espaço de decisão,
de 2 a 40. Como o orçamento acompanha a dimensão ($31D-1$), a pergunta não é se o problema fica mais
difícil — fica —, mas **quais famílias degradam mais rápido** e **quanto custa** cada ponto de
qualidade. O custo vem da camada ④ (tempo por fase: ajuste do modelo, busca da aquisição, sonda).

In [ ]:
tempo = pd.read_parquet(CACHE/'tempo_2026-08-17.parquet')
tempo = tempo[tempo.alg.isin(ALGORITMOS)].copy()
tempo['semente'] = tempo['semente'].astype(int)
tempo['grupo'] = tempo['alg'].map(GRUPO_ALG); tempo['nome'] = tempo['alg'].map(nome)
tempo['papel'] = tempo['alg'].map(df_alg['papel']); tempo['D'] = tempo['problema'].map(DIM)
tempo['fit_por_ger_s'] = tempo['fit_total_s'] / tempo['n_ger'].replace(0, np.nan)
print('células com tempo:', len(tempo), '| colunas:', [c for c in tempo.columns])
print('⚠ retrato de 17/08 — a camada ④ é a mesma do corpus congelado para as células já coletadas.')

In [ ]:
# (a) qualidade × D por família · (b) custo de ajuste × D por algoritmo
qual_D = (df[(df.regime=='online') & (df.tipo_problema=='sintético') & (df.grupo!='isolado')]
            .groupby(['grupo','D'])['hv_pct'].median().reset_index())
fitD = (tempo[(tempo.papel=='assistido')].groupby(['alg','nome','grupo','D'])['fit_por_ger_s']
          .median().reset_index().dropna())

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for g in [x for x in ORDEM_GRUPO if x in set(qual_D['grupo'])]:
    s = qual_D[qual_D.grupo==g].sort_values('D')
    axes[0].plot(s['D'], s['hv_pct'], 'o-', ms=5, color=COR_GRUPO[g], lw=2.0 if g!='piso' else 2.4,
                 ls='-' if g!='piso' else '--', label=g)
axes[0].set_xlabel('dimensão do espaço de decisão $D$'); axes[0].set_ylabel('HV alcançado (%)')
axes[0].set_title('(a) a degradação com a dimensão, por família', fontsize=10.5)
axes[0].legend(fontsize=8, frameon=False, ncol=2); axes[0].grid(color='#f2f2f2', lw=0.7)

for a in fitD.alg.unique():
    s = fitD[fitD.alg==a].sort_values('D')
    axes[1].plot(s['D'], s['fit_por_ger_s'], 'o-', ms=4, color=CORES.get(a), lw=1.5, label=nome(a), alpha=0.9)
axes[1].set_yscale('log'); axes[1].set_xlabel('dimensão do espaço de decisão $D$')
axes[1].set_ylabel('tempo de ajuste do modelo por geração (s)')
axes[1].set_title('(b) a parede do retreino', fontsize=10.5)
axes[1].legend(fontsize=6.5, frameon=False, ncol=2, loc='upper left'); axes[1].grid(color='#f2f2f2', lw=0.7)
for ax in axes: ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); salvar(fig, 'fig541_dimensionalidade'); plt.show()
gravar_tabela(qual_D.pivot(index='grupo', columns='D', values='hv_pct'), 'tab_64_qualidade_por_D')
gravar_tabela(fitD.pivot_table(index='nome', columns='D', values='fit_por_ger_s'), 'tab_65_fit_por_D')

In [ ]:
# Inclinação da degradação: quanto de HV cada família perde por unidade de log-D
inclin = []
for g, s in qual_D.groupby('grupo'):
    s = s.dropna()
    if len(s) < 4: continue
    b = np.polyfit(np.log2(s['D']), s['hv_pct'], 1)
    inclin.append({'grupo':g, 'perda_por_dobra_de_D': b[0], 'hv_em_D2': np.polyval(b, 1)})
inclin = pd.DataFrame(inclin).sort_values('perda_por_dobra_de_D')
print('Perda de HV (pontos percentuais) a cada vez que D dobra:')
display(inclin.round(2)); gravar_tabela(inclin, 'tab_66_inclinacao_D')

### 3.3 · Custo × benefício: o quadrante barato-e-bom

O tempo de parede por execução contra o rank médio de qualidade. A leitura é por quadrantes, não por
reta: o que interessa é quem entrega qualidade sem pagar a parede do processo gaussiano.

In [ ]:
custo = (tempo[tempo.problema.isin(PROBLEMAS_SINT)].groupby(['alg','nome','grupo','papel'])
           .agg(min_mediano=('wall_total_s','median'), fit_frac=('fit_total_s','sum'),
                busca_frac=('busca_total_s','sum'), wall=('wall_total_s','sum')).reset_index())
custo['min_mediano'] /= 60
rank_geral = (df[(df.regime=='online')&(df.tipo_problema=='sintético')]
                .groupby('alg')['rank_igd_plus'].mean())
custo['rank'] = custo['alg'].map(rank_geral)
custo = custo.dropna(subset=['rank','min_mediano'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.0), gridspec_kw={'width_ratios':[1.25,1]})
ax = axes[0]
for r in custo.itertuples():
    ax.scatter(r.min_mediano, r.rank, s=130, color=COR_GRUPO[r.grupo], alpha=0.92,
               marker='s' if r.papel=='piso' else 'o', edgecolor='k', lw=0.5)
    ax.annotate(r.nome, (r.min_mediano, r.rank), fontsize=7.5, xytext=(6,4), textcoords='offset points')
ax.set_xscale('log'); ax.invert_yaxis()
ax.axhline(custo['rank'].median(), color='#cccccc', lw=0.9, ls=':')
ax.axvline(custo['min_mediano'].median(), color='#cccccc', lw=0.9, ls=':')
ax.set_xlabel('tempo de parede mediano por execução (min, escala log)')
ax.set_ylabel('rank médio de IGD+ (menor = melhor)')
ax.set_title('(a) o custo se paga?', fontsize=10.5)
ax.text(0.02, 0.04, 'barato e bom', transform=ax.transAxes, fontsize=8.5, color='#1d6b3a', style='italic')
ax.text(0.72, 0.94, 'caro e fraco', transform=ax.transAxes, fontsize=8.5, color='#c2570a', style='italic')

# (b) decomposição do tempo por fase
dec = custo.set_index('nome')[['fit_frac','busca_frac','wall']].sort_values('wall', ascending=True)
dec['outro'] = (dec['wall'] - dec['fit_frac'] - dec['busca_frac']).clip(lower=0)
prop = dec[['fit_frac','busca_frac','outro']].div(dec['wall'], axis=0)*100
b = np.zeros(len(prop))
for col, cor, lab in [('fit_frac','#1f77b4','ajuste do modelo'), ('busca_frac','#ff7f0e','busca da aquisição'),
                      ('outro','#cccccc','avaliação e demais')]:
    axes[1].barh(range(len(prop)), prop[col], left=b, color=cor, alpha=0.9, label=lab); b += prop[col].values
axes[1].set_yticks(range(len(prop))); axes[1].set_yticklabels(prop.index, fontsize=8)
axes[1].set_xlabel('% do tempo de parede'); axes[1].set_xlim(0,100)
axes[1].set_title('(b) onde o tempo é gasto', fontsize=10.5)
axes[1].legend(fontsize=8, frameon=False, loc='lower right')
for a_ in axes: a_.spines[['top','right']].set_visible(False)
plt.tight_layout(); salvar(fig, 'fig542_custo_beneficio'); plt.show()
display(custo[['nome','grupo','min_mediano','rank']].sort_values('rank').round(2))
gravar_tabela(custo, 'tab_67_custo_beneficio')

---
# Parte 4 · O impacto do surrogate: a sonda

Nenhum trabalho do corpus valida a incerteza que consome: mede-se o erro da média, nunca o
$\sigma$ contra o erro real ao longo da busca. A **sonda** fecha essa lacuna. É um conjunto fixo de
pontos por problema — os mesmos para todos os algoritmos, sementes e gerações — que cada modelo
re-prediz a cada retreino, com os valores verdadeiros pré-computados **fora** do orçamento.

Três dimensões, sobre os mesmos pontos:

* **acurácia** — o erro quadrático médio da predição da média ($\mathrm{RMSE}$): o modelo aprende?
* **calibração de ordem** — a correlação de postos entre o $\sigma$ reportado e o erro cometido:
  quando o modelo diz que está inseguro, ele erra mais?
* **cobertura** — a fração dos pontos em que o valor verdadeiro cai dentro de $\mu \pm 2\sigma$,
  contra a taxa nominal de $95{,}4\%$: a banda tem a largura que promete?

In [ ]:
sonda = pd.read_parquet(CACHE/'sonda_2026-08-17.parquet')
sonda = sonda[sonda.alg.isin(ALGORITMOS)].copy()
sonda['semente'] = sonda['semente'].astype(int)
sonda['grupo'] = sonda['alg'].map(GRUPO_ALG); sonda['nome'] = sonda['alg'].map(nome)
sonda['medicao'] = sonda['alg'].map(df_alg['medicao']); sonda['surrogate'] = sonda['alg'].map(df_alg['surrogate'])
sonda['D'] = sonda['problema'].map(DIM)
# progresso do treino: fração do orçamento já consumida quando o modelo foi reajustado
sonda['frac_treino'] = sonda['fe_treino_max'] / sonda['problema'].map(ORC)
print('linhas da sonda:', len(sonda), '| configs:', sonda.alg.nunique(), '| problemas:', sonda.problema.nunique())
print('sem sonda (declarado):', sorted(set(PROBLEMAS) - set(sonda.problema)))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
# (a) acurácia ao longo do treino
ev = sonda.assign(bin=pd.cut(sonda.frac_treino, np.linspace(0,1,9), labels=False)).dropna(subset=['bin'])
for a in sorted(ev.alg.unique()):
    s = ev[ev.alg==a].groupby('bin')['rmse'].median()
    axes[0].plot((s.index+0.5)/8, s.values, 'o-', ms=3.5, color=CORES.get(a), lw=1.4, label=nome(a), alpha=0.9)
axes[0].set_yscale('log'); axes[0].set_xlabel('fração do orçamento no retreino')
axes[0].set_ylabel('erro de predição (RMSE)'); axes[0].set_title('(a) o modelo aprende?', fontsize=10.5)
axes[0].legend(fontsize=6, frameon=False, ncol=2)

# (b) calibração de ordem
cal = sonda.groupby(['alg','nome','grupo'])['rho_sigma_erro'].median().reset_index().sort_values('rho_sigma_erro')
axes[1].barh(range(len(cal)), cal['rho_sigma_erro'], color=[COR_GRUPO[g] for g in cal['grupo']], alpha=0.9)
axes[1].set_yticks(range(len(cal))); axes[1].set_yticklabels(cal['nome'], fontsize=8.5)
axes[1].axvline(0, color='k', lw=0.8)
axes[1].set_xlabel(r'$\rho$ de Spearman entre $\sigma$ e |erro|')
axes[1].set_title('(b) a incerteza ordena o erro?', fontsize=10.5)

# (c) cobertura contra o nominal
cob = sonda.groupby(['alg','nome','grupo'])['cobertura_2sigma'].median().reset_index().sort_values('cobertura_2sigma')
axes[2].barh(range(len(cob)), 100*cob['cobertura_2sigma'], color=[COR_GRUPO[g] for g in cob['grupo']], alpha=0.9)
axes[2].axvline(95.4, color='#c2570a', lw=1.3, ls='--')
axes[2].text(95.4, len(cob)-0.6, ' nominal 95,4%', fontsize=7.5, color='#c2570a', va='top')
axes[2].set_yticks(range(len(cob))); axes[2].set_yticklabels(cob['nome'], fontsize=8.5)
axes[2].set_xlabel(r'cobertura de $\mu \pm 2\sigma$ (%)')
axes[2].set_title('(c) a banda tem a largura que promete?', fontsize=10.5)
for ax in axes: ax.spines[['top','right']].set_visible(False); ax.grid(axis='x', color='#f2f2f2', lw=0.7)
plt.tight_layout(); salvar(fig, 'fig551_sonda'); plt.show()

tab_sonda = (sonda.groupby(['alg','nome','grupo','medicao','surrogate'])
               .agg(rmse=('rmse','median'), rho=('rho_sigma_erro','median'),
                    cobertura=('cobertura_2sigma','median')).reset_index().sort_values('rho', ascending=False))
display(tab_sonda.round(3)); gravar_tabela(tab_sonda, 'tab_68_sonda')

### 4.2 · A qualidade do modelo explica a qualidade da otimização?

A pergunta central desta dissertação, em forma testável: um surrogate mais acurado — ou uma incerteza
mais bem calibrada — produz uma otimização melhor? A unidade é a célula (configuração × problema ×
semente), e a comparação é **dentro** do problema, por rank.

In [ ]:
sq = (sonda.groupby(['alg','problema','semente'])
        .agg(rmse=('rmse','median'), rho=('rho_sigma_erro','median'), cob=('cobertura_2sigma','median'))
        .reset_index()
        .merge(df[['alg','problema','semente','igd_plus','hv_pct','rank_igd_plus','grupo','nome']],
               on=['alg','problema','semente'], how='inner'))
# rank do erro do modelo dentro do problema-semente: compara modelo com modelo
sq['rank_rmse'] = sq.groupby(['problema','semente'])['rmse'].rank()
print('células com sonda e métrica:', len(sq))

from scipy.stats import spearmanr
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.4))
pares = [('rank_rmse','rank_igd_plus','rank do erro do modelo','rank de IGD+','(a) modelo acurado → otimização boa?'),
         ('rho','rank_igd_plus',r'calibração de ordem ($\rho$)','rank de IGD+','(b) incerteza calibrada → otimização boa?'),
         ('cob','hv_pct',r'cobertura de $\mu\pm2\sigma$','HV alcançado (%)','(c) banda honesta → mais hipervolume?')]
for ax, (xc, yc, xl, yl, tt) in zip(axes, pares):
    s = sq[[xc,yc,'grupo','alg','nome']].dropna()
    agg = s.groupby(['alg','nome','grupo'])[[xc,yc]].median().reset_index()
    ax.scatter(s[xc], s[yc], s=5, c=[COR_GRUPO[g] for g in s['grupo']], alpha=0.10, lw=0)
    ax.scatter(agg[xc], agg[yc], s=110, c=[COR_GRUPO[g] for g in agg['grupo']], edgecolor='k', lw=0.5, zorder=3)
    for r in agg.itertuples(): ax.annotate(r.nome, (getattr(r,xc), getattr(r,yc)), fontsize=7, xytext=(5,3), textcoords='offset points')
    rho_, p_ = spearmanr(s[xc], s[yc])
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.set_title(f'{tt}\nρ = {vg(rho_)} · p {"< 0,001" if p_<1e-3 else "= "+vg(p_,3)} · n = {len(s)}', fontsize=9.5)
    ax.spines[['top','right']].set_visible(False); ax.grid(color='#f4f4f4', lw=0.7)
plt.tight_layout(); salvar(fig, 'fig552_surrogate_vs_otimizacao'); plt.show()

corrs = pd.DataFrame([{'relação':tt, 'rho_spearman':spearmanr(sq[xc].dropna(), sq.loc[sq[xc].notna(),yc])[0],
                       'n':int(sq[xc].notna().sum())} for xc,yc,_,_,tt in pares])
display(corrs.round(3)); gravar_tabela(corrs, 'tab_69_correlacao_sonda')

---
# Parte 5 · Os três problemas sobre dados reais

A escada de proveniência do Capítulo 4: o RE21, canônico da otimização estrutural com formulação
publicada; o DDMOP7, benchmark dirigido a dados com avaliador caixa-preta; e o ESTOQUE40, formulação
própria pré-registrada sobre dados públicos de varejo. Duas perguntas: **a suíte sintética prediz o
comportamento nos reais?** e **o que as melhores soluções significam**, no vocabulário de quem
decide.

Duas ressalvas de leitura declaradas: o DDMOP7 é reportado **somente** por hipervolume (não tem
frente de referência — D102.4) e o ESTOQUE40 tem cauda de execuções incompleta.

In [ ]:
reais = df[df.tipo_problema=='real'].copy()
print(reais.groupby(['problema','regime'])['alg'].nunique().to_string())
print('\ncélulas por problema:', reais.groupby('problema').size().to_dict())

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.8))
for ax, p in zip(axes, PROBLEMAS_REAIS):
    met = 'hv_pct' if p in SEM_FRONT else 'igd_plus'
    s = reais[(reais.problema==p) & (reais.regime=='online')]
    if s.empty: ax.set_visible(False); continue
    ordem = s.groupby('alg')[met].median().sort_values(ascending=(met!='hv_pct')).index.tolist()
    ordem = [a for a in ordem if s[s.alg==a][met].notna().any()]
    if not ordem: ax.set_visible(False); continue
    sns.boxplot(data=s[s.alg.isin(ordem)], y='alg', x=met, order=ordem, hue='alg', hue_order=ordem,
                palette={a: COR_GRUPO[GRUPO_ALG[a]] for a in ordem}, legend=False,
                ax=ax, width=0.68, fliersize=1.2, linewidth=0.7)
    ax.set_yticklabels([nome(a) for a in ordem], fontsize=8)
    ax.set_ylabel(''); ax.set_xlabel('HV alcançado (%)' if met=='hv_pct' else 'IGD+ (menor = melhor)')
    sufixo = ' · só hipervolume (sem frente de referência)' if p in SEM_FRONT else ''
    ax.set_title(f'{p}{sufixo}', fontsize=10)
    ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); salvar(fig, 'fig561_reais_comparativo'); plt.show()

In [ ]:
# A suíte sintética prediz o real? Correlação de postos entre o ranking sintético e o de cada real
rank_sint = (df[(df.regime=='online')&(df.tipo_problema=='sintético')&(df.papel=='assistido')]
               .groupby('alg')['rank_igd_plus'].mean().rename('sintetico'))
linhas = []
for p in PROBLEMAS_REAIS:
    met = 'hv' if p in SEM_FRONT else 'igd_plus'
    r = (reais[(reais.problema==p)&(reais.regime=='online')&(reais.papel=='assistido')]
           .groupby('alg')[met].median().rank(ascending=(met!='hv')))
    j = pd.concat([rank_sint, r.rename('real')], axis=1).dropna()
    if len(j) < 4: continue
    rho_, p_ = spearmanr(j['sintetico'], j['real'])
    linhas.append({'problema':p, 'rho':rho_, 'p_valor':p_, 'n_configs':len(j)})
transf = pd.DataFrame(linhas)
print('Transferência do ranking sintético → real (Spearman):'); display(transf.round(3))
gravar_tabela(transf, 'tab_70_transferencia_sintetico_real')

fig, axes = plt.subplots(1, len(PROBLEMAS_REAIS), figsize=(4.6*len(PROBLEMAS_REAIS), 4.2))
for ax, p in zip(np.atleast_1d(axes), PROBLEMAS_REAIS):
    met = 'hv' if p in SEM_FRONT else 'igd_plus'
    r = (reais[(reais.problema==p)&(reais.regime=='online')&(reais.papel=='assistido')]
           .groupby('alg')[met].median().rank(ascending=(met!='hv')))
    j = pd.concat([rank_sint, r.rename('real')], axis=1).dropna()
    if j.empty: ax.set_visible(False); continue
    ax.scatter(j['sintetico'], j['real'], s=110, c=[COR_GRUPO[GRUPO_ALG[a]] for a in j.index], edgecolor='k', lw=0.5)
    for a in j.index: ax.annotate(nome(a), (j.at[a,'sintetico'], j.at[a,'real']), fontsize=7, xytext=(5,3), textcoords='offset points')
    lim = [0, max(j.max())+1]; ax.plot(lim, lim, ':', color='#bbbbbb', lw=1)
    rr = transf.loc[transf.problema==p,'rho']
    ax.set_xlabel('rank médio nos 25 sintéticos'); ax.set_ylabel(f'rank no {p}')
    ax.set_title(f'{p} · ρ = {vg(rr.iloc[0]) if len(rr) else "—"}', fontsize=10)
    ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); salvar(fig, 'fig562_sintetico_prediz_real'); plt.show()

### 5.2 · O que as melhores soluções significam

A otimização não entrega um projeto: entrega o conjunto não dominado, um catálogo de compromissos. Os
painéis abaixo mostram, para cada problema real, a frente encontrada pela execução de desempenho
mediano do melhor algoritmo, nas **unidades do problema** — e não em objetivo normalizado.

In [ ]:
UNIDADES = {
 'RE21':      (('volume de material (cm³)', 'deslocamento do nó (cm)'), (1,1)),
 'ESTOQUE40': (('receita realizada (£, negada)', 'custo de estoque parado (£)'), (1,1)),
 'DDMOP7':    (('complexidade: pesos não nulos / 17', 'erro de classificação: casos / 690'), (17,690)),
}
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.6))
for ax, p in zip(axes, PROBLEMAS_REAIS):
    d = nd(p); s = reais[(reais.problema==p)&(reais.regime=='online')&(reais.papel=='assistido')]
    if d is None or s.empty: ax.set_visible(False); continue
    met = 'hv' if p in SEM_FRONT else 'igd_plus'
    melhor = s.groupby('alg')[met].median().sort_values(ascending=(met!='hv')).index[0]
    ss = s[s.alg==melhor]
    sem = int(ss.iloc[(ss[met]-ss[met].median()).abs().argsort().iloc[0]]['semente'])
    g = d[(d.alg==melhor)&(d.semente==sem)]
    fs = cols_f(g); (xl, yl), (kx, ky) = UNIDADES[p]
    Ft = frentes.loc[frentes.problema==p].dropna(axis=1)
    if not Ft.empty and len(cols_f(Ft))==2:
        ax.scatter(Ft[cols_f(Ft)[0]]*kx, Ft[cols_f(Ft)[1]]*ky, s=5, c='#dddddd', label='frente de referência')
    ax.scatter(g[fs[0]]*kx, g[fs[1]]*ky, s=34, color=CORES.get(melhor), edgecolor='k', lw=0.4,
               label=f'{nome(melhor)} (execução mediana)', zorder=3)
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(f'{p} · {len(g)} soluções não dominadas', fontsize=10)
    ax.legend(fontsize=7.5, frameon=False); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); salvar(fig, 'fig563_reais_solucoes'); plt.show()

In [ ]:
# Os extremos e o meio-termo de cada frente real, em unidades do problema — o catálogo do decisor
cat = []
for p in PROBLEMAS_REAIS:
    d = nd(p); s = reais[(reais.problema==p)&(reais.regime=='online')&(reais.papel=='assistido')]
    if d is None or s.empty: continue
    met = 'hv' if p in SEM_FRONT else 'igd_plus'
    melhor = s.groupby('alg')[met].median().sort_values(ascending=(met!='hv')).index[0]
    ss = s[s.alg==melhor]; sem = int(ss.iloc[(ss[met]-ss[met].median()).abs().argsort().iloc[0]]['semente'])
    g = d[(d.alg==melhor)&(d.semente==sem)]; fs = cols_f(g)
    F = g[fs].to_numpy(float); (xl, yl), (kx, ky) = UNIDADES[p]
    ideal, nadir = MET.reference_bounds(p); Fn = MET.normalize(F, ideal, nadir)
    idx = {'extremo em $f_1$': int(F[:,0].argmin()), 'extremo em $f_2$': int(F[:,1].argmin()),
           'meio-termo (joelho)': int(np.linalg.norm(Fn[:,:2], axis=1).argmin())}
    for rot, i in idx.items():
        cat.append({'problema':p, 'algoritmo':nome(melhor), 'solução':rot,
                    xl.split(' (')[0]: F[i,0]*kx, yl.split(' (')[0]: F[i,1]*ky})
cat = pd.DataFrame(cat)
print('O catálogo de compromissos: extremos e joelho de cada frente real')
display(cat.round(4)); gravar_tabela(cat, 'tab_71_solucoes_reais')

---
# Parte 6 · Testes estatísticos: o que tem confiança e o que é oscilação

As Partes 1 a 5 relatam medianas e placares — leitura **descritiva**. Esta parte separa o que
sobrevive a um teste do que cabe dentro da margem de oscilação entre sementes. Três instrumentos,
na convenção dominante do corpus:

1. **Wilcoxon pareado por sementes** de cada assistido contra o seu piso casado, problema a problema,
   com correção de Holm dentro de cada algoritmo — responde "este algoritmo bate o piso, mesmo?".
2. **Friedman + Nemenyi** sobre os postos de todos os algoritmos nos problemas completos, com o
   diagrama de diferença crítica — responde "existe diferença real no plantel, e entre quem?".
3. **Piso empírico de ruído**: a diferença mediana entre duas metades aleatórias das sementes do
   **mesmo** algoritmo no mesmo problema. Diferenças abaixo dele são oscilação, não resultado.

In [ ]:
from scipy.stats import wilcoxon, friedmanchisquare, rankdata
from statistics import NormalDist

def holm(ps):
    """Correção de Holm-Bonferroni; devolve os p ajustados na ordem original."""
    ps = np.asarray(ps, float); o = np.argsort(ps); m = len(ps)
    aj = np.empty(m); corr = 0.0
    for i, k in enumerate(o):
        corr = max(corr, (m-i)*ps[k]); aj[k] = min(corr, 1.0)
    return aj

ALPHA = 0.05
linhas = []
for a in [x for x in ASSISTIDOS if x in set(df.alg)]:
    ref = PISO_CASADO[a]; regime = 'offline' if a in OFFLINES else 'online'
    d = df[df.regime==regime]
    pw = d[d.papel=='piso'].pivot_table(index=['problema','semente'], columns='alg', values='igd_plus')
    if ref == 'BANDA':
        base_ = pw[[p for p in PISOS if p in pw and p!='moead_media']].min(axis=1)
    elif ref in pw: base_ = pw[ref]
    else: continue
    s = d[d.alg==a].set_index(['problema','semente'])['igd_plus']
    j = pd.concat({'alg':s,'piso':base_}, axis=1).dropna()
    for p, g in j.groupby(level=0):
        if len(g) < 6: continue
        try: st, pv = wilcoxon(g['alg'], g['piso'])
        except ValueError: continue
        linhas.append({'alg':a,'nome':nome(a),'grupo':GRUPO_ALG[a],'ref':ref,'problema':p,
                       'n':len(g), 'mediana_alg':g['alg'].median(), 'mediana_piso':g['piso'].median(),
                       'delta':g['piso'].median()-g['alg'].median(), 'p':pv})
wx = pd.DataFrame(linhas)
wx['p_holm'] = wx.groupby('alg')['p'].transform(lambda s: holm(s.values))
wx['veredito'] = np.select([(wx.p_holm<ALPHA)&(wx.delta>0), (wx.p_holm<ALPHA)&(wx.delta<=0)],
                           ['vence o piso','perde do piso'], default='indistinguível')
print(wx.veredito.value_counts().to_string())
gravar_tabela(wx, 'tab_72_wilcoxon_vs_piso')

In [ ]:
res_wx = (wx.pivot_table(index=['nome','grupo'], columns='veredito', values='problema', aggfunc='count')
            .fillna(0).astype(int).reindex(columns=['vence o piso','indistinguível','perde do piso'], fill_value=0)
            .sort_values('vence o piso', ascending=False).reset_index())

fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.4), gridspec_kw={'width_ratios':[1,1.15]})
ax = axes[0]; b = np.zeros(len(res_wx))
for col, cor in [('vence o piso','#1d6b3a'), ('indistinguível','#d9d9d9'), ('perde do piso','#c2570a')]:
    ax.barh(range(len(res_wx)), res_wx[col], left=b, color=cor, alpha=0.92, label=col); b += res_wx[col].values
ax.set_yticks(range(len(res_wx))); ax.set_yticklabels(res_wx['nome'], fontsize=8.5); ax.invert_yaxis()
ax.set_xlabel(f'problemas (Wilcoxon pareado, Holm, α = {vg(ALPHA)})')
ax.set_title('(a) o que sobrevive ao teste, contra o piso casado', fontsize=10.5)
ax.legend(fontsize=8, frameon=False, loc='lower right'); ax.spines[['top','right','left']].set_visible(False)

# (b) Friedman + diagrama de diferença crítica sobre os problemas completos
comp = (df[(df.regime=='online')&(df.tipo_problema=='sintético')]
          .pivot_table(index='problema', columns='alg', values='igd_plus', aggfunc='median').dropna(axis=1, how='any'))
comp = comp.dropna()
st, pv = friedmanchisquare(*[comp[c].values for c in comp.columns])
R = pd.Series(np.apply_along_axis(rankdata, 1, comp.values).mean(axis=0), index=comp.columns).sort_values()
k, N = comp.shape[1], comp.shape[0]
q_alpha = NormalDist().inv_cdf(1 - ALPHA/(k*(k-1)))*np.sqrt(2)/np.sqrt(2)   # aprox. de Nemenyi via normal
CD = q_alpha*np.sqrt(k*(k+1)/(6*N))
ax = axes[1]
ax.hlines(range(len(R)), 1, R.values, color='#dddddd', lw=1)
ax.scatter(R.values, range(len(R)), s=95, c=[COR_GRUPO[GRUPO_ALG[a]] for a in R.index], edgecolor='k', lw=0.5, zorder=3)
ax.set_yticks(range(len(R))); ax.set_yticklabels([nome(a) for a in R.index], fontsize=8.5); ax.invert_yaxis()
ax.axvspan(R.values[0], R.values[0]+CD, color='#1d6b3a', alpha=0.10)
ax.set_xlabel('rank médio de IGD+ (menor = melhor)')
ax.set_title(f'(b) Friedman: p {"< 0,001" if pv<1e-3 else "= "+vg(pv,3)} · '
             f'diferença crítica = {vg(CD)} · {N} problemas', fontsize=10.5)
ax.spines[['top','right','left']].set_visible(False)
plt.tight_layout(); salvar(fig, 'fig571_testes'); plt.show()
display(res_wx); gravar_tabela(res_wx, 'tab_73_veredito_por_algoritmo')
print(f'\nFriedman: χ² = {st:.2f} · p = {pv:.3g} · k = {k} algoritmos · N = {N} problemas · CD = {CD:.3f}')

In [ ]:
# O piso empírico de ruído: quanto duas metades das sementes do MESMO algoritmo diferem entre si
rng = np.random.default_rng(20260928)
ruido = []
for (a, p), g in df[(df.regime=='online')&(df.tipo_problema=='sintético')].groupby(['alg','problema']):
    v = g['igd_plus'].dropna().values
    if len(v) < 10: continue
    for _ in range(40):
        idx = rng.permutation(len(v)); h = len(v)//2
        ruido.append(abs(np.median(v[idx[:h]]) - np.median(v[idx[h:2*h]])) / max(np.median(v), 1e-12))
piso_ruido = float(np.quantile(ruido, 0.95))
print(f'Piso empírico de ruído (percentil 95 da diferença relativa entre metades): {100*piso_ruido:.2f}%')

wx['delta_rel'] = (wx['delta'] / wx['mediana_piso'].abs().clip(lower=1e-12)).abs()
wx['acima_do_ruido'] = wx['delta_rel'] > piso_ruido
resumo_conf = (wx.groupby('veredito')
                 .agg(comparacoes=('problema','size'), acima_do_ruido=('acima_do_ruido','sum')))
resumo_conf['%_acima'] = (100*resumo_conf['acima_do_ruido']/resumo_conf['comparacoes']).round(1)
print('\nQuais diferenças superam a oscilação entre sementes:'); display(resumo_conf)
gravar_tabela(resumo_conf, 'tab_74_piso_de_ruido')

---
## Fecho · O que este notebook entrega

Cada figura gravada em `dissertacao/figures/capitulo5/` tem, ao lado, a tabela que a sustenta em
`ua-dd-saea/data/analysis_cache/cap5_tabelas/`. Nenhum número vai ao texto sem sair de uma dessas
tabelas — e toda tabela nasce da execução desta sessão sobre o corpus congelado.

⚠ **Pendências declaradas, para o crivo do autor:** (1) as réguas de teste — família de testes,
correção de multiplicidade e uso do piso de ruído como filtro — são decisão dele (d44–d47); o que
está aqui é a convenção dominante do corpus, aplicada como proposta. (2) A matriz de características
dos problemas carrega o selo D98. (3) As camadas de sonda, tempo, trajetória e infill vêm do retrato
de 17/08 do repositório experimental — as células que entraram depois não estão nelas.

In [ ]:
figs = sorted(p.name for p in FIGDIR.glob('*.pdf'))
tabs = sorted(p.name for p in TABDIR.glob('*.csv'))
print(f'{len(figs)} figuras em figures/capitulo5/:'); [print('  ', f) for f in figs]
print(f'\n{len(tabs)} tabelas de proveniência em data/analysis_cache/cap5_tabelas/')

### Exportação das tabelas para o capítulo

Cada tabela de proveniência vira um `tabular` em `dissertacao/tables/capitulo5/`, que o `.tex` do
capítulo puxa por `\input`. Nenhum número é digitado à mão no texto: a cadeia vai do artefato bruto
à página, sem passo manual no meio.

In [ ]:
# Exporta as tabelas do capítulo (CSV de proveniência) para LaTeX.
import pandas as pd, numpy as np, re
from pathlib import Path
UA   = Path('/Users/gmello/Documents/python_repos/mestrado/ua-dd-saea')
DISS = Path('/Users/gmello/Documents/python_repos/mestrado/dissertacao')
TAB  = UA/'data/analysis_cache/cap5_tabelas'
OUT  = DISS/'tables/capitulo5'; OUT.mkdir(parents=True, exist_ok=True)

NOME_COL = {  # cabeçalhos legíveis, em português, sem vocabulário de bastidor
 'nome':'Algoritmo','grupo':'Família','familia':'Família','ref':'Régua','V':'V','D':'D','N':'N',
 'delta_mediano':'$\\Delta$ mediano','raia':'Raia','degrau_falha':'Degrau de falha','degraus':'Degraus',
 'vs_piso_primeiro':'vs.\\ piso (1.º)','vs_piso_chefe':'vs.\\ piso (chefe)','queda':'Queda',
 'igdx':'IGDX','cobertura':'Modais cobertos','igdp':'IGD$^{+}$','par':'Par','problema':'Problema',
 'delta':'$\\Delta$ IGD$^{+}$','vitorias':'Vitórias','problemas':'Problemas',
 'perda_por_dobra_de_D':'Perda por dobra de $D$','hv_em_D2':'HV em $D=2$',
 'min_mediano':'Tempo (min)','rank':'Rank','medicao':'Medição','surrogate':'Substituto',
 'rmse':'Erro (RMSE)','rho':'$\\rho(\\sigma,|e|)$','relação':'Relação','rho_spearman':'$\\rho$ de Spearman',
 'n':'$n$','p_valor':'$p$','n_configs':'Configs','algoritmo':'Algoritmo','solução':'Solução',
 'vence o piso':'Vence o piso','indistinguível':'Indistinguível','perde do piso':'Perde do piso',
 'comparacoes':'Comparações','acima_do_ruido':'Acima do ruído','%_acima':'\\% acima','veredito':'Veredito',
}
def cab(c):
    c = str(c)
    return NOME_COL.get(c, c.replace('_',' ').capitalize())

def fmt(v):
    if isinstance(v, float):
        if np.isnan(v): return '---'
        s = f'{v:,.3f}' if abs(v) < 10 else f'{v:,.1f}'
        return s.replace(',', '@').replace('.', ',').replace('@', '.')
    s = str(v)
    # escapa só o que ainda não vem escapado (evita o '\\%' duplo, que o TeX lê como quebra de linha)
    for c in '&%_#': s = re.sub(r'(?<!\\)' + re.escape(c), '\\' + c, s)
    return s.replace('nan','---')

def tex(df, arq, col_fmt=None, index=False, max_rows=None):
    d = df.copy()
    d = d.drop(columns=[c for c in d.columns if str(c).startswith('Unnamed:')], errors='ignore')
    if max_rows: d = d.head(max_rows)
    if index: d = d.reset_index()
    cols = list(d.columns)
    align = col_fmt or ('l'*1 + 'c'*(len(cols)-1))
    linhas = ['\\begin{tabular}{' + align + '}', '\\hline',
              ' & '.join('\\agente{\\textbf{'+cab(c)+'}}' for c in cols) + ' \\\\', '\\hline']
    for _, r in d.iterrows():
        linhas.append(' & '.join('\\agente{'+fmt(v)+'}' for v in r.values) + ' \\\\')
    linhas += ['\\hline', '\\end{tabular}']
    (OUT/arq).write_text('\n'.join(linhas) + '\n', encoding='utf-8')
    print(f'→ tables/capitulo5/{arq}  ({len(d)}×{len(cols)})')

def le(nome, **kw):
    p = TAB/f'{nome}.csv'
    return pd.read_csv(p, **kw) if p.exists() else None

# ── tab 57 · degrau de falha (pivot família × raia) ─────────────────────────
d = le('tab_57_degrau_falha')
if d is not None:
    piv = d.pivot(index='grupo', columns='raia', values='degrau_falha').fillna('---')
    piv.columns = [c[:16] for c in piv.columns]
    tex(piv.reset_index().rename(columns={'grupo':'familia'}), 'tab_57_degrau_falha.tex',
        col_fmt='l'+'c'*len(piv.columns))

# ── tab 58 · placar contra o piso ───────────────────────────────────────────
d = le('tab_58_placar_pisos')
if d is not None:
    d = d[['nome','grupo','ref','V','D','delta_mediano']].sort_values('V', ascending=False)
    d['ref'] = d['ref'].replace({'BANDA':'banda dos pisos','nsga2':'NSGA-II','nsga3':'NSGA-III',
                                 'moead':'MOEA/D','smsemoa':'SMS-EMOA','moead_media':'MOEA/D-média'})
    tex(d, 'tab_58_placar_pisos.tex', col_fmt='llcccr')

# ── tab 61 · IGDX por algoritmo ─────────────────────────────────────────────
d = le('tab_61_igdx_por_algoritmo')
if d is not None:
    d = d[['nome','grupo','igdx','cobertura','igdp']].sort_values('igdx')
    d['cobertura'] = (100*d['cobertura']).round(0).astype(int).astype(str) + '%'
    tex(d, 'tab_61_igdx_por_algoritmo.tex', col_fmt='llccc')

# ── tab 63 · contrastes ─────────────────────────────────────────────────────
d = le('tab_63_contrastes')
if d is not None:
    g = (d.groupby('par').agg(vitorias=('delta', lambda s:(s>0).sum()),
                              problemas=('delta','size'), delta=('delta','median')).reset_index())
    g['par'] = g['par'].str.replace('\n', ' --- ', regex=False)
    tex(g, 'tab_63_contrastes.tex', col_fmt='p{6.6cm}ccr')

# ── tab 66 · inclinação com D ───────────────────────────────────────────────
d = le('tab_66_inclinacao_D')
if d is not None:
    tex(d[['grupo','perda_por_dobra_de_D','hv_em_D2']].sort_values('perda_por_dobra_de_D'),
        'tab_66_inclinacao_D.tex', col_fmt='lcc')

# ── tab 67 · custo × benefício ──────────────────────────────────────────────
d = le('tab_67_custo_beneficio')
if d is not None:
    tex(d[['nome','grupo','min_mediano','rank']].sort_values('rank'),
        'tab_67_custo_beneficio.tex', col_fmt='llrc')

# ── tab 68 · sonda ──────────────────────────────────────────────────────────
d = le('tab_68_sonda')
if d is not None:
    d = d[['nome','grupo','surrogate','medicao','rmse','rho','cobertura']].sort_values('rho', ascending=False)
    d['cobertura'] = (100*d['cobertura']).round(1).map(lambda v: f'{v:.1f}'.replace('.',',')+'%')
    tex(d, 'tab_68_sonda.tex', col_fmt='llccccc')

# ── tab 69 · correlação sonda × otimização ──────────────────────────────────
d = le('tab_69_correlacao_sonda')
if d is not None:
    d['relação'] = d['relação'].str.replace(r'^\([a-c]\)\s*', '', regex=True)
    tex(d[['relação','rho_spearman','n']], 'tab_69_correlacao_sonda.tex', col_fmt='p{8.4cm}cc')

# ── tab 70 · transferência sintético → real ─────────────────────────────────
d = le('tab_70_transferencia_sintetico_real')
if d is not None:
    tex(d[['problema','rho','p_valor','n_configs']], 'tab_70_transferencia_sintetico_real.tex', col_fmt='lccc')

# ── tab 71 · soluções reais ─────────────────────────────────────────────────
d = le('tab_71_solucoes_reais')
if d is not None:
    d = d.fillna('')
    tex(d, 'tab_71_solucoes_reais.tex', col_fmt='ll' + 'l'*(len(d.columns)-2))

# ── tab 73 · veredito por algoritmo ─────────────────────────────────────────
d = le('tab_73_veredito_por_algoritmo')
if d is not None:
    cols = [c for c in ['nome','grupo','vence o piso','indistinguível','perde do piso'] if c in d.columns]
    tex(d[cols], 'tab_73_veredito_por_algoritmo.tex', col_fmt='ll'+'c'*(len(cols)-2))

# ── tab 74 · piso de ruído ──────────────────────────────────────────────────
d = le('tab_74_piso_de_ruido', index_col=0)
if d is not None:
    tex(d, 'tab_74_piso_de_ruido.tex', col_fmt='lccc', index=True)

print('\nexportadas:', len(list(OUT.glob('*.tex'))), 'tabelas')